# preprocessing shallow five years

Sanitised source snapshot. See the repository README and WORKFLOW.md before running. Outputs and execution metadata are removed. Historical experiments are not an unattended pipeline or reproduced scientific result.


In [ ]:
import os
from pathlib import Path

REPO_ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "prepare_wells.py").is_file())
WORKSPACE_ROOT = Path(os.environ.get("CGWB_WORKSPACE", str(REPO_ROOT / "data" / "cgwb"))).expanduser().resolve()
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE_ROOT)


In [ ]:
type_of_aquifer='shallow'

In [ ]:
import os

print("Current working directory:", os.getcwd())


# calculated data

In [ ]:
import pandas as pd

In [ ]:
# Define the file path
file_path = f"{WORKSPACE_ROOT}/18. preprocessing_shallow_aquifers/input/WellDepths_shallow_aquifers_CSV.csv"

# Read the CSV file
shallow_aquifers_calculated_df = pd.read_csv(file_path)

# Preview the first few rows
print(shallow_aquifers_calculated_df.head())

In [ ]:
shallow_aquifers_calculated_df

In [ ]:
for column in shallow_aquifers_calculated_df.columns:
  print(column)

In [ ]:
shallow_aquifers_calculated_df['2001-01-01'][333]

In [ ]:
import pandas as pd
import json

In [ ]:
# Step 1: Copy the original DataFrame
shallow_aquifers_renamed_df = shallow_aquifers_calculated_df.copy()

# Step 2: Define metadata columns that should be skipped
metadata_cols = ['system:index', 'microwatershed_id', 'weighted_avg_yeild', '.geo']

# Step 3: Find columns with JSON data
json_columns = [col for col in shallow_aquifers_renamed_df.columns if col not in metadata_cols]

# Step 4: Rename "WellDepth" to "ChangeinWellDepth_Calculated" in JSON cells
def rename_welldepth_key(cell):
    try:
        data = json.loads(cell)
        if 'WellDepth' in data:
            data['CWD_Calculated'] = data.pop('WellDepth')
        return json.dumps(data)
    except:
        return cell  # If not JSON, return as-is

for col in json_columns:
    shallow_aquifers_renamed_df[col] = shallow_aquifers_renamed_df[col].apply(rename_welldepth_key)

# Step 5: Rename column names from '2001-01' to '2001_Jan_March' and so on
month_to_season = {
    '01': 'Jan_March',
    '04': 'April_June',
    '07': 'Jul_Sept',
    '10': 'Oct_Dec'
}

rename_dict = {}
for col in shallow_aquifers_renamed_df.columns:
    if col[:4].isdigit() and col[4] == '-' and col[5:7] in month_to_season:
        year = col[:4]
        month = col[5:7]
        season = month_to_season[month]
        rename_dict[col] = f"{year}_{season}"

shallow_aquifers_renamed_df.rename(columns=rename_dict, inplace=True)

# Step 6: Print a few renamed columns as a sample
print("✅ Renamed columns (sample):")
print([col for col in shallow_aquifers_renamed_df.columns if '_' in col and col.count('_') == 1][:10])

In [ ]:
shallow_aquifers_renamed_df

In [ ]:
shallow_aquifers_renamed_df['2001_Jan_March'][0]

In [ ]:
import pandas as pd
import json
import json
import re

In [ ]:
# Step 1: Create a copy
df = shallow_aquifers_renamed_df.copy()

# Step 2: Identify all seasonal columns
pattern = r"(\d{4})_(Jan_March|April_June|Jul_Sept|Oct_Dec)"
seasonal_cols = [col for col in df.columns if re.match(pattern, col)]

# Step 3: Map season name to number
season_order = {
    "Jan_March": 1,
    "April_June": 2,
    "Jul_Sept": 3,
    "Oct_Dec": 4
}

# Step 4: Melt to long format
id_vars = ['system:index', 'microwatershed_id', 'weighted_avg_yeild', '.geo']
df_melted = df.melt(id_vars=id_vars, value_vars=seasonal_cols,
                    var_name='year_season', value_name='json_data')

# Step 5: Extract year and season from column name
df_melted[['year', 'season_name']] = df_melted['year_season'].str.extract(r'(\d{4})_(Jan_March|April_June|Jul_Sept|Oct_Dec)')
df_melted['season_number'] = df_melted['season_name'].map(season_order).astype(int)
df_melted['year'] = df_melted['year'].astype(int)

# Step 6: Parse JSON safely
def parse_json(val):
    try:
        if isinstance(val, str):
            return json.loads(val)
        elif isinstance(val, dict):
            return val
    except:
        return {}
    return {}

parsed_df = df_melted['json_data'].apply(parse_json).apply(pd.Series)

# Step 7: Combine all together
shallow_aquifer_unrolled = pd.concat([df_melted[id_vars + ['year', 'season_name', 'season_number']], parsed_df], axis=1)

In [ ]:
shallow_aquifer_unrolled

In [ ]:
# Copy original data
df = shallow_aquifer_unrolled.copy()
df['year'] = df['year'].astype(int)

# Covariates to average
covariates = ['DeltaG', 'ET', 'G', 'Precipitation', 'RunOff', 'CWD_Calculated']
max_year = df['year'].max()


# Container for results
lagged_data = []

for lag in range(0, 6):
    for mws_id in df['microwatershed_id'].unique():
        mws_data = df[df['microwatershed_id'] == mws_id]

        for start_year in mws_data['year'].unique():
            end_year = start_year + lag
            if end_year <= max_year:
                target_years = list(range(start_year, end_year + 1))

                subset = mws_data[mws_data['year'].isin(target_years)].copy()

                # Sort subset by year and season order
                subset = subset.sort_values(by=['year', 'season_number'])

                if (subset['year'].nunique() == (lag + 1)) and subset[covariates].notna().all().all():
                    mean_vals = subset[covariates].mean(numeric_only=True)

                    row_info = subset.iloc[0][['microwatershed_id', 'weighted_avg_yeild', '.geo', 'year']]
                    combined = pd.concat([row_info, mean_vals])
                    combined['lag'] = lag
                    lagged_data.append(combined)

# Final DataFrame
lagged_covariate_df_strict = pd.DataFrame(lagged_data).reset_index(drop=True)

In [ ]:
lagged_covariate_df_strict

In [ ]:
# Check for columns with any NaNs
missing_values = lagged_covariate_df_strict.isna().sum()

# Display only those columns that contain missing values
print(missing_values[missing_values > 0])

In [ ]:
lagged_covariate_df_strict

In [ ]:
shallow_aquifer_unrolled[shallow_aquifer_unrolled['microwatershed_id']=='001300000000000078b5']

In [ ]:
lagged_covariate_df_strict[
    (lagged_covariate_df_strict['microwatershed_id'] == '001300000000000078b5') &
    (lagged_covariate_df_strict['year'] == 2001)]

In [ ]:
for column in lagged_covariate_df_strict.columns:
  print(column)

*take only 5 year lag*

In [ ]:
lag_5_df = lagged_covariate_df_strict[lagged_covariate_df_strict['lag'] == 5]

In [ ]:
lag_5_df

# terrain and stream order

In [ ]:
file_path = f'{WORKSPACE_ROOT}/6. Adding Stream order and terrain information/input/final_filtered_data_for_geometry_micro_watershed.csv'
# Read CSV into a DataFrame
terrain_filt_df = pd.read_csv(file_path)

In [ ]:
terrain_filt_df

In [ ]:
for column in terrain_filt_df.columns:
  print(column)

In [ ]:
# Count NaNs in terrain_cluster column
nan_count = terrain_filt_df['terrain_cluster'].isna().sum()
print("Number of NaNs in terrain_cluster:", nan_count)

# Optional: Boolean check
print("No NaNs present? ->", nan_count == 0)

In [ ]:
import ast
import json
import pandas as pd

In [ ]:
# Sets of IDs
terrain_ids = set(terrain_filt_df['id_micro_watershed'])
final_ids = set(lag_5_df['microwatershed_id'])

# Check if all terrain IDs exist in final_df
terrain_in_final = terrain_ids.issubset(final_ids)
print("All terrain_filt_df IDs in final_df? ->", terrain_in_final)

# Check if all final_df IDs exist in terrain_filt_df
final_in_terrain = final_ids.issubset(terrain_ids)
print("All final_df IDs in terrain_filt_df? ->", final_in_terrain)

# Find missing IDs
missing_in_final = terrain_ids - final_ids
missing_in_terrain = final_ids - terrain_ids

# Print counts
print("Number of IDs in terrain_filt_df not in final_df:", len(missing_in_final))
print("Number of IDs in final_df not in terrain_filt_df:", len(missing_in_terrain))

In [ ]:
# Perform a left join on 'microwatershed_id' to bring in 'terrain_cluster'
cwd_with_terrain_df = lag_5_df.merge(
    terrain_filt_df[['id_micro_watershed', 'terrain_cluster']],
    how='left',
    left_on='microwatershed_id',
    right_on='id_micro_watershed'
)

# Drop the redundant right-side join key
cwd_with_terrain_df = cwd_with_terrain_df.drop(columns=['id_micro_watershed'])

In [ ]:
cwd_with_terrain_df

In [ ]:
for column in cwd_with_terrain_df:
  print(column)

In [ ]:
# Drop rows where terrain_cluster is NaN
cwd_with_terrain_df = cwd_with_terrain_df.dropna(subset=['terrain_cluster']).reset_index(drop=True)

In [ ]:
cwd_with_terrain_df

In [ ]:
# Count NaN values in terrain_cluster
nan_count = cwd_with_terrain_df['terrain_cluster'].isna().sum()
print("Number of NaNs in terrain_cluster:", nan_count)

# Optional: Boolean check
print("No NaNs present? ->", nan_count == 0)

*stream order*

In [ ]:
import pandas as pd

# Define the file path for reading the CSV
csv_file_path = f'{WORKSPACE_ROOT}/6. Adding Stream order and terrain information/input/final_filtered_data_for_geo_mws_so_area.csv'

# Read the CSV file into a DataFrame
stream_order_continous = pd.read_csv(csv_file_path)

# Display the first few rows of the DataFrame to ensure it was read correctly
print(stream_order_continous)

In [ ]:
for column in stream_order_continous.columns:
  print(column)

In [ ]:
# Select only the stream order columns (excluding id and geometry)
stream_order_columns = [col for col in stream_order_continous.columns if col not in ['id_micro_w', 'geometry']]

# Perform the merge
cwd_with_terrain_and_stream_order_df = cwd_with_terrain_df.merge(
    stream_order_continous[['id_micro_w'] + stream_order_columns],
    how='left',
    left_on='microwatershed_id',
    right_on='id_micro_w'
).drop(columns=['id_micro_w'])

In [ ]:
cwd_with_terrain_and_stream_order_df

In [ ]:
for column in cwd_with_terrain_and_stream_order_df.columns:
  print(column)

In [ ]:
# List of columns to check
stream_order_check_columns = [
    'Area%_SO_1', 'Area%_SO_2', 'Area%_SO_3', 'Area%_SO_4', 'Area%_SO_5',
    'Area%_SO_6', 'Area%_SO_7', 'Area%_SO_8', 'Area%_SO_9', 'Area%_SO_10', 'Area%_SO_11'
]

# Check for NaNs and print count per column
nan_counts = cwd_with_terrain_and_stream_order_df[stream_order_check_columns].isna().sum()
print("NaN counts per column:\n", nan_counts)

# Optional: Check if all columns are fully non-null
all_non_null = (nan_counts == 0).all()
print("\nAll columns have no NaNs? ->", all_non_null)

In [ ]:
lag_5_df

In [ ]:
for column in lag_5_df.columns:
  print(column)

In [ ]:
cwd_with_terrain_and_stream_order_df

*add soge*

In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame
soge_shallow = pd.read_csv(f'{WORKSPACE_ROOT}/21. Soge shallow/output/soge_shallow.csv')

In [ ]:
soge_shallow

In [ ]:
for column in soge_shallow.columns:
  print(column)

In [ ]:
soge_shallow.rename(
    columns={col: col.replace("sgw_dev_pe_", "soge_") for col in soge_shallow.columns if col.startswith("sgw_dev_pe_")},
    inplace=True
)

In [ ]:
soge_shallow

In [ ]:
# Melt the DataFrame
soge_shallow_melted = soge_shallow.melt(
    id_vars='microwatershed_id',
    var_name='year',
    value_name='soge'
)

# Extract year from the 'year' column (e.g., from 'soge_2013' → '2013')
soge_shallow_melted['year'] = soge_shallow_melted['year'].str.extract(r'soge_(\d{4})')

In [ ]:
soge_shallow_melted

In [ ]:
cwd_with_terrain_and_stream_order_df

In [ ]:
# Step 1: Rename 'soge' to 'soge_t_minus_1' in the melted DataFrame
soge_shallow_melted_renamed = soge_shallow_melted.rename(columns={'soge': 'soge_t_minus_1'})

# Step 2: Add 1 to 'year' so it aligns with year t in the main dataframe (originally it was t-1)
soge_shallow_melted_renamed['year'] = pd.to_numeric(
    soge_shallow_melted_renamed['year'], errors='coerce'
) + 1

In [ ]:
soge_shallow_melted_renamed

In [ ]:
# Step 3: Merge on microwatershed_id and year
final_dataframe = cwd_with_terrain_and_stream_order_df.merge(
    soge_shallow_melted_renamed,
    how='left',
    on=['microwatershed_id', 'year'],
)

In [ ]:
final_dataframe

In [ ]:
nan_count = final_dataframe['soge_t_minus_1'].isna().sum()
print(f"Number of rows with NaN in soge_t_minus_1: {nan_count}")

In [ ]:
final_dataframe = final_dataframe.dropna(subset=['soge_t_minus_1'])

In [ ]:
final_dataframe

*add surface water presence*

In [ ]:
import pandas as pd

# Read the CSV file into the DataFrame
surface_water_presence_shallow = pd.read_csv(
    f'{WORKSPACE_ROOT}/18. preprocessing_shallow_aquifers/input/CropSeason_AllYears_Combined_shallow.csv'
)

In [ ]:
surface_water_presence_shallow

In [ ]:
for column in surface_water_presence_shallow.columns:
  print(column)

In [ ]:
# List of columns to retain
columns_to_keep = [
    'krkrz_by_total_pct_2017', 'krkrz_by_total_pct_2018', 'krkrz_by_total_pct_2019',
    'krkrz_by_total_pct_2020', 'krkrz_by_total_pct_2021', 'krkrz_by_total_pct_2022',
    'krz_by_total_pct_2017', 'krz_by_total_pct_2018', 'krz_by_total_pct_2019',
    'krz_by_total_pct_2020', 'krz_by_total_pct_2021', 'krz_by_total_pct_2022',
    'microwatershed_id', '.geo'# Optional: keep for merging later
]

# Subset the DataFrame
surface_water_presence_shallow = surface_water_presence_shallow[columns_to_keep]

In [ ]:
surface_water_presence_shallow

In [ ]:
for column in surface_water_presence_shallow.columns:
  print(column)

In [ ]:
# Step 1: Melt krkrz_by_total_pct columns
krkrz_melted = surface_water_presence_shallow.melt(
    id_vars='microwatershed_id',
    value_vars=[f'krkrz_by_total_pct_{year}' for year in range(2017, 2023)],  # ← stop at 2022
    var_name='year',
    value_name='krkrz_by_total_pct'
)

krkrz_melted['year'] = krkrz_melted['year'].str.extract(r'(\d{4})').astype(int)

# Step 2: Melt krz_by_total_pct columns
krz_melted = surface_water_presence_shallow.melt(
    id_vars='microwatershed_id',
    value_vars=[f'krz_by_total_pct_{year}' for year in range(2017, 2023)],  # ← stop at 2022
    var_name='year',
    value_name='krz_by_total_pct'
)

krz_melted['year'] = krz_melted['year'].str.extract(r'(\d{4})').astype(int)

# Step 3: Merge both melted DataFrames on microwatershed_id and year
surface_water_presence_shallow_melted = pd.merge(
    krkrz_melted,
    krz_melted,
    on=['microwatershed_id', 'year'],
    how='inner'
)

In [ ]:
surface_water_presence_shallow_melted

In [ ]:
# Merge surface water metrics into final_dataframe
final_dataframe = final_dataframe.merge(
    surface_water_presence_shallow_melted,
    how='left',
    on=['microwatershed_id', 'year']
)

In [ ]:
final_dataframe

In [ ]:
for column in final_dataframe.columns:
  print(column)

In [ ]:
final_dataframe['year']

In [ ]:
# For krkrz_by_total_pct
krkrz_nan_count = final_dataframe['krkrz_by_total_pct'].isna().sum()
krkrz_not_nan_count = final_dataframe['krkrz_by_total_pct'].notna().sum()

# For krz_by_total_pct
krz_nan_count = final_dataframe['krz_by_total_pct'].isna().sum()
krz_not_nan_count = final_dataframe['krz_by_total_pct'].notna().sum()

# Print results
print("krkrz_by_total_pct → NaN:", krkrz_nan_count, "| Not NaN:", krkrz_not_nan_count)
print("krz_by_total_pct   → NaN:", krz_nan_count, "| Not NaN:", krz_not_nan_count)


In [ ]:
final_dataframe = final_dataframe.dropna(subset=['krkrz_by_total_pct', 'krz_by_total_pct']).reset_index(drop=True)

In [ ]:
final_dataframe

In [ ]:
for column in final_dataframe.columns:
  print(column)

*mws new to old mapping*

In [ ]:
file_path = f'{WORKSPACE_ROOT}/23. Training_shallow_five_years/input/shallow_mws_mapping.csv'
# Read CSV
new_to_old_id_mapping = pd.read_csv(file_path)

In [ ]:
new_to_old_id_mapping

In [ ]:
# Step 1: Merge surface water info with final_dataframe
final_dataframe_after_id_mapping = final_dataframe.merge(
    new_to_old_id_mapping[['microwatershed_id', 'uid']],
    on='microwatershed_id',
    how='left'
)

In [ ]:
final_dataframe_after_id_mapping

*adding max stream order*

In [ ]:
import pandas as pd

# Correctly handle spaces in file path
file_path = f"{WORKSPACE_ROOT}/23. Training_shallow_five_years/input/max_stream_order_shallow.csv"

# Read CSV
max_stream_order = pd.read_csv(file_path)

In [ ]:
max_stream_order

In [ ]:
# Step 1: Merge surface water info with final_dataframe
final_dataframe_after_max_stream_order = final_dataframe_after_id_mapping.merge(
    max_stream_order[['microwatershed_id', 'max_stream_order']],
    on='microwatershed_id',
    how='left'
)

In [ ]:
final_dataframe_after_max_stream_order

*adding connectivity info and catchment area*

In [ ]:
import pandas as pd

# Correctly handle spaces in file path
file_path = f"{WORKSPACE_ROOT}/23. Training_shallow_five_years/input/catchment_area_connectivity_Info.csv"

# Read CSV
connectivity_catchment_area = pd.read_csv(file_path)

In [ ]:
connectivity_catchment_area

In [ ]:
# Step 1: Merge surface water info with final_dataframe
final_dataframe_after_connectivity_catchment = final_dataframe_after_max_stream_order.merge(
    connectivity_catchment_area[['uid', 'mws_connectivity_indegree','summed_upstream_catchment_area']],
    on='uid',
    how='left'
)

In [ ]:
final_dataframe_after_connectivity_catchment

*adding cropping percentage*

In [ ]:
import pandas as pd

# Correctly handle spaces in file path
file_path = f"{WORKSPACE_ROOT}/23. Training_shallow_five_years/input/Cropping_AllYears_Combined_shallow.csv"

# Read CSV
cropping_percentage = pd.read_csv(file_path)

In [ ]:
cropping_percentage

In [ ]:
for column in cropping_percentage.columns:
  print(column)

In [ ]:
# List of columns to retain
columns_to_keep = ['cropping_krkrz_by_total_pct_2017',
'cropping_krkrz_by_total_pct_2018',
'cropping_krkrz_by_total_pct_2019',
'cropping_krkrz_by_total_pct_2020',
'cropping_krkrz_by_total_pct_2021',
'cropping_krkrz_by_total_pct_2022',
'cropping_krz_by_total_pct_2017',
'cropping_krz_by_total_pct_2018',
'cropping_krz_by_total_pct_2019',
'cropping_krz_by_total_pct_2020',
'cropping_krz_by_total_pct_2021',
'cropping_krz_by_total_pct_2022',
    'microwatershed_id', '.geo'# Optional: keep for merging later
]

# Subset the DataFrame
cropping_percentage = cropping_percentage[columns_to_keep]

In [ ]:
# Step 1: Melt krkrz_by_total_pct columns
cropping_krkrz_melted = cropping_percentage.melt(
    id_vars='microwatershed_id',
    value_vars=[f'cropping_krkrz_by_total_pct_{year}' for year in range(2017, 2023)],  # ← stop at 2022
    var_name='year',
    value_name='cropping_krkrz_by_total_pct'
)

cropping_krkrz_melted['year'] = cropping_krkrz_melted['year'].str.extract(r'(\d{4})').astype(int)

# Step 2: Melt krz_by_total_pct columns
cropping_krz_melted = cropping_percentage.melt(
    id_vars='microwatershed_id',
    value_vars=[f'cropping_krz_by_total_pct_{year}' for year in range(2017, 2023)],  # ← stop at 2022
    var_name='year',
    value_name='cropping_krz_by_total_pct'
)

cropping_krz_melted['year'] = cropping_krz_melted['year'].str.extract(r'(\d{4})').astype(int)

# Step 3: Merge both melted DataFrames on microwatershed_id and year
cropping_percentage_melted = pd.merge(
    cropping_krkrz_melted,
    cropping_krz_melted,
    on=['microwatershed_id', 'year'],
    how='inner'
)

In [ ]:
cropping_percentage_melted

In [ ]:
# Merge surface water metrics into final_dataframe
final_dataframe_after_cropping_percentage = final_dataframe_after_connectivity_catchment.merge(
    cropping_percentage_melted,
    how='left',
    on=['microwatershed_id', 'year']
)

In [ ]:
final_dataframe_after_cropping_percentage

In [ ]:
final_dataframe_after_cropping_percentage = final_dataframe_after_cropping_percentage.dropna()

In [ ]:
final_dataframe_after_cropping_percentage

*infiltration rate*

In [ ]:
file_path = f"{WORKSPACE_ROOT}/23. Training_shallow_five_years/input/infiltration_combined.csv"
infiltration_rate = pd.read_csv(file_path)

In [ ]:
infiltration_rate

In [ ]:
# Drop rows with duplicate 'Code' values (keep first occurrence only)
infiltration_rate_cleaned = infiltration_rate.drop_duplicates(subset='Code', keep='first')

# Rename the 'Recommended Rate' column to 'infiltration_rate'
infiltration_rate_cleaned = infiltration_rate_cleaned.rename(columns={'Recommended Rate': 'infiltration_rate'})
infiltration_rate_cleaned = infiltration_rate_cleaned.rename(columns={'Code': 'newcode43'})

# Display the cleaned DataFrame
print(infiltration_rate_cleaned.head())

In [ ]:
infiltration_rate_cleaned

# get the filtered list of mws along with their aquifer category

In [ ]:
import pandas as pd

# Path to your pickle file
file_path = f"{WORKSPACE_ROOT}/5. Thresholding MWS aquifer wise/output/final_filtered_data_after_threshold.pickle"

# Load the pickle file as a DataFrame
thresholded_mws_ids_with_aquifer_types = pd.read_pickle(file_path)

# Display the first few rows
print(thresholded_mws_ids_with_aquifer_types.head())

In [ ]:
thresholded_mws_ids_with_aquifer_types

In [ ]:
for column in thresholded_mws_ids_with_aquifer_types.columns:
    print(column)

In [ ]:
thresholded_mws_ids_with_aquifer_types[
    ['aquifer', 'aquifer0', 'aquifers', 'area_re', 'avg_mbgl', 'm2_perday', 'm3_per_day', 'mbgl', 'newcode14', 'newcode43']
]

In [ ]:
thresholded_mws_ids_with_aquifer_types['aquifer_category'].isnull().sum()

# removing unnecessary columns

In [ ]:
# List of columns to drop
columns_to_drop = [
    'objectid_left', 'pointInRegion', 'station_id', 'station_name', 'index_right', 'id_right',
    'aquifer', 'aquifer0', 'aquifers', 'area_re', 'avg_mbgl', 'm2_perday', 'm3_per_day', 'mbgl',
    'newcode14',
    #'newcode43',
    'objectid_right', 'pa_order', 'per_cm', 'shape_Area', 'shape_Leng',
    'state', 'system', 'test', 'yeild__', 'zone_m','index_right_mws','id_left','id','DN'
]

# Create a new DataFrame without the unwanted columns
filtered_thresholded_mws_ids_with_aquifer_types = thresholded_mws_ids_with_aquifer_types.drop(columns=columns_to_drop, errors='ignore')

# View the result
print(filtered_thresholded_mws_ids_with_aquifer_types.head())

In [ ]:
filtered_thresholded_mws_ids_with_aquifer_types

In [ ]:
# --- Case 2: Flip specific quarterly columns from 2001–2023
quarter_cols = [
    "2001_Jan_March","2001_April_June","2001_Jul_Sept","2001_Oct_Dec",
    "2002_Jan_March","2002_April_June","2002_Jul_Sept","2002_Oct_Dec",
    "2003_Jan_March","2003_April_June","2003_Jul_Sept","2003_Oct_Dec",
    "2004_Jan_March","2004_April_June","2004_Jul_Sept","2004_Oct_Dec",
    "2005_Jan_March","2005_April_June","2005_Jul_Sept","2005_Oct_Dec",
    "2006_Jan_March","2006_April_June","2006_Jul_Sept","2006_Oct_Dec",
    "2007_Jan_March","2007_April_June","2007_Jul_Sept","2007_Oct_Dec",
    "2008_Jan_March","2008_April_June","2008_Jul_Sept","2008_Oct_Dec",
    "2009_Jan_March","2009_April_June","2009_Jul_Sept","2009_Oct_Dec",
    "2010_Jan_March","2010_April_June","2010_Jul_Sept","2010_Oct_Dec",
    "2011_Jan_March","2011_April_June","2011_Jul_Sept","2011_Oct_Dec",
    "2012_Jan_March","2012_April_June","2012_Jul_Sept","2012_Oct_Dec",
    "2013_Jan_March","2013_April_June","2013_Jul_Sept","2013_Oct_Dec",
    "2014_Jan_March","2014_April_June","2014_Jul_Sept","2014_Oct_Dec",
    "2015_Jan_March","2015_April_June","2015_Jul_Sept","2015_Oct_Dec",
    "2016_Jan_March","2016_April_June","2016_Jul_Sept","2016_Oct_Dec",
    "2017_Jan_March","2017_April_June","2017_Jul_Sept","2017_Oct_Dec",
    "2018_Jan_March","2018_April_June","2018_Jul_Sept","2018_Oct_Dec",
    "2019_Jan_March","2019_April_June","2019_Jul_Sept","2019_Oct_Dec",
    "2020_Jan_March","2020_April_June","2020_Jul_Sept","2020_Oct_Dec",
    "2021_Jan_March","2021_April_June","2021_Jul_Sept","2021_Oct_Dec",
    "2022_Jan_March","2022_April_June","2022_Jul_Sept","2022_Oct_Dec",
    "2023_Jan_March","2023_April_June","2023_Jul_Sept","2023_Oct_Dec"
]

# Make sure only existing columns are selected
quarter_cols = [c for c in quarter_cols if c in filtered_thresholded_mws_ids_with_aquifer_types.columns]
filtered_thresholded_mws_ids_with_aquifer_types[quarter_cols] = filtered_thresholded_mws_ids_with_aquifer_types[quarter_cols].mul(-1)

In [ ]:
filtered_thresholded_mws_ids_with_aquifer_types

*changing aquifer categories to alluvial and non alluvial*

In [ ]:
filtered_thresholded_mws_ids_with_aquifer_types['aquifer_category'] = filtered_thresholded_mws_ids_with_aquifer_types['aquifer_category'].apply(
    lambda x: 'alluvial' if x == 'Alluvial' else 'non_alluvial'
)

In [ ]:
for column in filtered_thresholded_mws_ids_with_aquifer_types.columns:
  print(column)

In [ ]:
import pandas as pd

In [ ]:
# Define season order
season_order = {
    'Jan_March': 1,
    'April_June': 2,
    'Jul_Sept': 3,
    'Oct_Dec': 4
}

# Step 1: Extract all valid year-season columns (those with 'YYYY_Season' format)
year_season_cols = [
    col for col in filtered_thresholded_mws_ids_with_aquifer_types.columns
    if '_' in col and col.split('_')[0].isdigit()
]

# Step 2: Unroll/melt the DataFrame
melted_df = filtered_thresholded_mws_ids_with_aquifer_types.melt(
    id_vars=['id_micro_watershed', 'aquifer_category', 'geometry','newcode43'],
    value_vars=year_season_cols,
    var_name='year_season',
    value_name='WellDepth'
)

# Step 3: Extract year and season
melted_df['year'] = melted_df['year_season'].apply(lambda x: int(x.split('_')[0]))
melted_df['season'] = melted_df['year_season'].apply(lambda x: '_'.join(x.split('_')[1:]))

# Step 4: Assign season number
melted_df['season_number'] = melted_df['season'].map(season_order)

# Step 5: Sort by microwatershed, year, season
melted_df = melted_df.sort_values(by=['id_micro_watershed', 'year', 'season_number']).reset_index(drop=True)

# Step 6: Rename column for consistency
melted_df = melted_df.rename(columns={'id_micro_watershed': 'microwatershed_id'})

# Remove non-numeric & NaN values
melted_df['WellDepth'] = pd.to_numeric(melted_df['WellDepth'], errors='coerce')
melted_df = melted_df.dropna(subset=['WellDepth'])

In [ ]:
melted_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
# Plotting
plt.figure(figsize=(20, 10))
sns.boxplot(data=melted_df, x='year_season', y='WellDepth', hue='aquifer_category')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Season & Year')
plt.ylabel('Water Level')
plt.title('Boxplot of seasonal water Levels by Aquifer Category')
plt.legend(title='Aquifer Category')
plt.tight_layout()
plt.show()

In [ ]:
# Drop unnecessary columns and rename as requested
melted_df = melted_df.drop(columns=['year_season'])

In [ ]:
melted_df

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import pandas as pd
from scipy.stats import linregress

In [ ]:
from joblib import Parallel, delayed
from scipy.stats import theilslopes
import pymannkendall as mk

In [ ]:
melted_df

In [ ]:
melted_df = melted_df[
    (melted_df['microwatershed_id'].isin(final_dataframe_after_cropping_percentage['microwatershed_id'])) &
    (melted_df['year'] >= 2017)
]

In [ ]:
melted_df

In [ ]:
# Merge melted_df with infiltration_rate_cleaned on newcode43 and Code
melted_df = melted_df.merge(
    infiltration_rate_cleaned[['newcode43', 'infiltration_rate']],
    on='newcode43',
    how='left'
)


# View the result
print(melted_df.head())

In [ ]:
melted_df

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import theilslopes
import pymannkendall as mk

In [ ]:
filtered_thresholded_mws_ids_with_aquifer_types

In [ ]:
for column in filtered_thresholded_mws_ids_with_aquifer_types.columns:
  print(column)

In [ ]:
# Copy original DataFrame
df = filtered_thresholded_mws_ids_with_aquifer_types.copy()

# Ensure years are sorted and unique
years = sorted(set(melted_df['year']))

# Convert all seasonal columns to numeric
for year in years:
    for season in ['Jan_March', 'April_June', 'Jul_Sept', 'Oct_Dec']:
        col = f"{year}_{season}"
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# Precompute year pairs (start <= end)
year_pairs = [(start, end) for start in years for end in years if end >= start]

# Store results
diff_results = []

# Add tqdm to iterrows
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Microwatershed Processing"):
    row_dict = row.to_dict()  # Convert once for faster access

    for start_year, end_year in year_pairs:
        # Fast skip: if no valid value in end_year's 4 seasons
        if all(pd.isna(row_dict.get(f"{end_year}_{season}")) for season in ['Jan_March', 'April_June', 'Jul_Sept', 'Oct_Dec']):
            continue

        # Construct column names from start to end year
        col_names = [
            f"{y}_{season}"
            for y in range(start_year, end_year + 1)
            for season in ['Jan_March', 'April_June', 'Jul_Sept', 'Oct_Dec']
        ]

        # Collect valid index-value pairs
        valid_pairs = [(i, row_dict[col]) for i, col in enumerate(col_names) if col in row_dict and pd.notna(row_dict[col])]
        if len(valid_pairs) < 2:
            continue

        # Unzip for slope and Mann-Kendall
        x_vals, y_vals = zip(*valid_pairs)
        try:
            slope, *_ = theilslopes(y_vals, x_vals)
            mk_result = mk.original_test(y_vals)
        except Exception:
            continue  # Safely skip problematic calculations

        diff_results.append({
            'microwatershed_id': row_dict['id_micro_watershed'],
            'aquifer_category': row_dict.get('aquifer_category'),
            'start_year': start_year,
            'end_year': end_year,
            'lag': end_year - start_year,
            'sens_slope': slope,
            'mk_trend': mk_result.trend,
            'mk_p_value': mk_result.p,
            'mk_tau': mk_result.Tau,
            'num_points': len(y_vals),
            'latitude': row_dict.get('latitude'),
            'longitude': row_dict.get('longitude'),
            'geometry': row_dict.get('geometry')
        })

# Convert to DataFrame
diff_sens_combined_df = pd.DataFrame(diff_results)

In [ ]:
diff_sens_combined_df

In [ ]:
# Ensure both columns are numeric
diff_sens_combined_df['sens_slope'] = pd.to_numeric(diff_sens_combined_df['sens_slope'], errors='coerce')
diff_sens_combined_df['lag'] = pd.to_numeric(diff_sens_combined_df['lag'], errors='coerce')

# Create the new column
diff_sens_combined_df['magnitude_of_slope'] = diff_sens_combined_df['sens_slope'] * diff_sens_combined_df['lag']

In [ ]:
diff_sens_combined_df

In [ ]:
# Group by aquifer_category and lag, then count the rows
lag_counts = diff_sens_combined_df.groupby(['aquifer_category', 'lag']).size().reset_index(name='count')

# Display the result
print(lag_counts)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))
sns.boxplot(data=diff_sens_combined_df, x='lag', y='magnitude_of_slope', hue='aquifer_category')
plt.title("Distribution of Magnitude of Slope by Lag and Aquifer Category")
plt.xlabel("Lag (years)")
plt.ylabel("Magnitude of Slope")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Filter out lag 0 and rows with null trend
mk_valid = diff_sens_combined_df[
    (diff_sens_combined_df['lag'] > 0) &
    (diff_sens_combined_df['mk_trend'].notna())
]

# Get unique lags
unique_lags = sorted(mk_valid['lag'].unique())

# Set up subplot grid
fig, axes = plt.subplots(len(unique_lags), 1, figsize=(6, len(unique_lags) * 4))

# Plot pie chart for each lag
for idx, lag in enumerate(unique_lags):
    ax = axes[idx]
    subset = mk_valid[mk_valid['lag'] == lag]
    counts = subset['mk_trend'].value_counts()
    labels = [f"{k} ({v})" for k, v in counts.items()]
    ax.pie(counts, labels=labels, autopct='%1.1f%%', startangle=140)
    ax.set_title(f"Trend Distribution for Lag {lag} years")

plt.tight_layout()
plt.show()

In [ ]:
diff_sens_combined_df.dtypes

*take only lag of 5 years*

In [ ]:
lag_5_df = diff_sens_combined_df[diff_sens_combined_df['lag'] == 5]

In [ ]:
lag_5_df

In [ ]:
lag_5_df.loc[:, 'longitude'] = lag_5_df['geometry'].apply(lambda point: point.x)
lag_5_df.loc[:, 'latitude'] = lag_5_df['geometry'].apply(lambda point: point.y)

In [ ]:
lag_5_df

In [ ]:
for column in lag_5_df.columns:
  print(column)

In [ ]:
# Merge melted_df with infiltration_rate_cleaned on newcode43 and Code
lag_5_df = lag_5_df.merge(
    melted_df[['microwatershed_id', 'infiltration_rate']],
    on='microwatershed_id',
    how='left'
)

*grace data addition*

In [ ]:
import pandas as pd
import glob

# Path where your 7 CSVs are saved
# (adjust the path to your folder)
path = f"{WORKSPACE_ROOT}/23. Training_shallow_five_years/input/"

# Pattern to match all the GRACE CSVs
all_files = glob.glob(path + "GRACE_EWT_AllMicrowatersheds_*.csv")

# Read and concatenate
df_list = [pd.read_csv(f) for f in all_files]
df = pd.concat(df_list, ignore_index=True)

print("Shape:", df.shape)
print("Columns:", df.columns)
print(df.head())

In [ ]:
import pandas as pd

# List your files in the exact order you want
files_ordered = [
    "GRACE_EWT_AllMicrowatersheds_2017_2018_shallow.csv",
    "GRACE_EWT_AllMicrowatersheds_2018_2019_shallow.csv",
    "GRACE_EWT_AllMicrowatersheds_2019_2020_shallow.csv",
    "GRACE_EWT_AllMicrowatersheds_2020_2021_shallow.csv",
    "GRACE_EWT_AllMicrowatersheds_2021_2022_shallow.csv",
    "GRACE_EWT_AllMicrowatersheds_2022_2023_shallow.csv",
    "GRACE_EWT_AllMicrowatersheds_2023_2024_shallow.csv"
]

# Adjust path if needed
path = f"{WORKSPACE_ROOT}/23. Training_shallow_five_years/input/"

# Read, rename, and concat
df_list = []
for f in files_ordered:
    temp = pd.read_csv(path + f)
    temp = temp.rename(columns={"mean": "ewt_cm"})
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

print("Final shape:", df.shape)
print(df.columns)
print(df.head())


In [ ]:
df

In [ ]:
import pandas as pd

# Assume df is already loaded and 'mean' renamed to 'ewt_cm'
# And df has 'date' and 'microwatershed_id'

# Ensure date is datetime
df['date'] = pd.to_datetime(df['date'])

# Extract year from date
df['year'] = df['date'].dt.year

# Step 1: set start_year, end_year, lag=0 for all existing data
df['start_year'] = df['year']
df['end_year'] = df['year']
df['lag'] = 0

# Step 2: generate lag=5 data
lagged_rows = []
for start in range(df['year'].min(), df['year'].max()-4):
    end = start + 5
    # only if both years exist in dataset
    if start in df['year'].values and end in df['year'].values:
        temp = df[(df['year'] == start) | (df['year'] == end)].copy()
        # group by microwatershed_id + month (ignoring year)
        temp['month'] = temp['date'].dt.month
        grouped = temp.groupby(['microwatershed_id','month'], as_index=False)['ewt_cm'].mean()
        grouped['start_year'] = start
        grouped['end_year'] = end
        grouped['lag'] = 5
        lagged_rows.append(grouped)

# Concatenate lag=5 results
if lagged_rows:
    df_lag5 = pd.concat(lagged_rows, ignore_index=True)
else:
    df_lag5 = pd.DataFrame()

# Step 3: combine back into one DataFrame
df_final = pd.concat([df, df_lag5], ignore_index=True, sort=False)

print(df_final.head())
print(df_final.tail())


In [ ]:
df_final

In [ ]:
import pandas as pd

# List your files in the exact order you want

files_ordered = [
    "GRACE_EWT_AllMicrowatersheds_2017_2018_shallow.csv",
    "GRACE_EWT_AllMicrowatersheds_2018_2019_shallow.csv",
    "GRACE_EWT_AllMicrowatersheds_2019_2020_shallow.csv",
    "GRACE_EWT_AllMicrowatersheds_2020_2021_shallow.csv",
    "GRACE_EWT_AllMicrowatersheds_2021_2022_shallow.csv",
    "GRACE_EWT_AllMicrowatersheds_2022_2023_shallow.csv",
    "GRACE_EWT_AllMicrowatersheds_2023_2024_shallow.csv"
]

# Adjust path if needed
path = f"{WORKSPACE_ROOT}/23. Training_shallow_five_years/input/"



# 2) Read, rename, and tag with file_start/file_end (parsed from filename)
dfs = []
for fname in files_ordered:
    m = re.search(r"(\d{4})_(\d{4})", fname)
    assert m, f"Could not parse years from filename: {fname}"
    fstart, fend = int(m.group(1)), int(m.group(2))

    tmp = pd.read_csv(path + fname)
    tmp = tmp.rename(columns={"mean": "ewt_cm"})  # ensure consistent name
    # Make sure date is datetime (if present)
    if "date" in tmp.columns:
        tmp["date"] = pd.to_datetime(tmp["date"], errors="coerce")
        tmp["year"] = tmp["date"].dt.year
        tmp["month"] = tmp["date"].dt.month
    else:
        # If no date column, you must have year/month columns already; assert they exist
        assert {"year","month"}.issubset(tmp.columns), "Need date or (year, month) columns"

    tmp["file_start"] = fstart
    tmp["file_end"] = fend
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)

In [ ]:
import pandas as pd

# df columns expected: microwatershed_id, date, ewt_cm  (or 'mean' to be renamed)
df = df.copy()

# Make sure we have ewt_cm
if 'ewt_cm' not in df.columns and 'mean' in df.columns:
    df = df.rename(columns={'mean': 'ewt_cm'})

# Types
df['date'] = pd.to_datetime(df['date'])
df['ewt_cm'] = pd.to_numeric(df['ewt_cm'], errors='coerce')
df['year'] = df['date'].dt.year

# -------------------------
# LAG = 0  (yearly means)
# -------------------------
yearly = (
    df.groupby(['microwatershed_id','year'], as_index=False)['ewt_cm']
      .mean()
)

lag0 = yearly.copy()
lag0['start_year'] = lag0['year']
lag0['end_year']   = lag0['year']
lag0['lag']        = 0
lag0 = lag0[['microwatershed_id','ewt_cm','start_year','end_year','lag']]

# ----------------------------------------------------------
# LAG = 5 (mean over ALL months from start_year..end_year)
# Only build a window if BOTH start_year and end_year exist
# ----------------------------------------------------------
years_available = set(df['year'].unique())
start_years = sorted(y for y in years_available if (y + 5) in years_available)

lag5_list = []
for s in start_years:
    e = s + 5
    # Filter to the full inclusive window [s .. e]
    window = df[(df['year'] >= s) & (df['year'] <= e)]
    # Average all monthly values in the window per MWS
    grouped = (window.groupby('microwatershed_id', as_index=False)['ewt_cm']
                      .mean())
    grouped['start_year'] = s
    grouped['end_year']   = e
    grouped['lag']        = 5
    lag5_list.append(grouped)

lag5 = pd.concat(lag5_list, ignore_index=True) if lag5_list else pd.DataFrame(
    columns=['microwatershed_id','ewt_cm','start_year','end_year','lag']
)

# -------------------------
# Combine & tidy
# -------------------------
df_out = pd.concat([lag0, lag5], ignore_index=True, sort=False)
df_out = df_out.sort_values(['microwatershed_id','lag','start_year']).reset_index(drop=True)

# Remove 'year' if present (from earlier steps)
df_out = df_out.drop(columns=['year'], errors='ignore')

print(df_out.head(12))
print(df_out.tail(12))


In [ ]:
df_out

In [ ]:
df

In [ ]:
import numpy as np
from scipy.stats import theilslopes

# helper: Sen's slope for one window (use monthly time index directly)
def sen_slope_for_window(mws_id, start_year, end_year):
    start_for_slope = max(2018, start_year)  # per your rule
    sub = df[(df['microwatershed_id'] == mws_id) &
             (df['year'] >= start_for_slope) &
             (df['year'] <= end_year)].copy()
    sub = sub.dropna(subset=['ewt_cm']).sort_values('date')
    if len(sub) < 2:
        return np.nan  # need at least 2 points

    # x = numeric month index (0,1,2,...), y = ewt_cm values
    t_idx = np.arange(len(sub))
    y = sub['ewt_cm'].astype(float).values

    slope, intercept, lo, hi = theilslopes(y, t_idx, 0.95)
    return float(slope)

# compute slope only for lag=5 rows
lag5_mask = (df_out['lag'] == 5)
df_out.loc[lag5_mask, 'grace_slope'] = df_out.loc[lag5_mask].apply(
    lambda r: sen_slope_for_window(
        r['microwatershed_id'], int(r['start_year']), int(r['end_year'])
    ),
    axis=1
)

# lag=0 rows: set NaN
df_out.loc[~lag5_mask, 'grace_slope'] = np.nan

print(df_out.head(12))
print(df_out[df_out['lag'] == 5].head(12))


In [ ]:
df_out

In [ ]:
df_out = df_out.dropna(subset=['grace_slope']).reset_index(drop=True)

In [ ]:
df_out

In [ ]:
# Make sure lag_5_df has the same key columns
keys = ['microwatershed_id', 'start_year', 'end_year', 'lag']

lag_5_df_with_grace = lag_5_df.merge(
    df_out.loc[df_out['lag']==5, keys + ['ewt_cm','grace_slope']],
    on=keys, how='left'
)

print(lag_5_df_with_grace.head())

In [ ]:
lag_5_df_with_grace = lag_5_df_with_grace.dropna(how='any').reset_index(drop=True)

print(lag_5_df_with_grace.shape)
print(lag_5_df_with_grace.head())


#training prep

In [ ]:
lag_5_df_with_grace

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_percentage_error
import matplotlib.pyplot as plt

In [ ]:
# Ensure datatypes match
lag_5_df_with_grace['start_year'] = lag_5_df_with_grace['start_year'].astype(int)
lag_5_df_with_grace['lag'] = lag_5_df_with_grace['lag'].astype(int)

final_dataframe_after_cropping_percentage['year'] = final_dataframe_after_cropping_percentage['year'].astype(int)
final_dataframe_after_cropping_percentage['lag'] = final_dataframe_after_cropping_percentage['lag'].astype(int)

# Merge on microwatershed_id, year, and lag
final_df = lag_5_df_with_grace.merge(
    final_dataframe_after_cropping_percentage,
    left_on=['microwatershed_id', 'start_year', 'lag'],
    right_on=['microwatershed_id', 'year', 'lag'],
    how='inner'  # or 'left' depending on what you want
)

In [ ]:
final_df

In [ ]:
has_nans = final_df.isna().any().any()
print("Contains NaNs:", has_nans)

# season wise model

In [ ]:
# Filter by aquifer category
shallow_alluvial = final_df[final_df['aquifer_category'] == 'alluvial'].copy()
shallow_non_alluvial = final_df[final_df['aquifer_category'] == 'non_alluvial'].copy()

In [ ]:
shallow_alluvial

*train test split based on mws id*

In [ ]:
shallow_alluvial['lag'].unique()

In [ ]:
# Separate by season (assuming a 'season' column with values 1 to 4)
def split_by_lag(df):
    return {
        5: df[df['lag'] == 5].copy(),
}

alluvial_lag = split_by_lag(shallow_alluvial)
non_alluvial_lag = split_by_lag(shallow_non_alluvial)

In [ ]:
alluvial_lag

In [ ]:
non_alluvial_lag

In [ ]:
for column in non_alluvial_lag[5].columns:
  print(column)

In [ ]:
save_path = f"{WORKSPACE_ROOT}/23. Training_shallow_five_years/output/"

# Save shallow alluvial
shallow_alluvial.to_csv(save_path + "shallow_alluvial.csv", index=False)

# Save shallow non-alluvial
shallow_non_alluvial.to_csv(save_path + "shallow_non_alluvial.csv", index=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Determine appropriate range based on your data
min_slope_alluvial = alluvial_lag[5]['magnitude_of_slope'].min()
max_slope_alluvial = alluvial_lag[5]['magnitude_of_slope'].max()
min_slope_non_alluvial = non_alluvial_lag[5]['magnitude_of_slope'].min()
max_slope_non_alluvial = non_alluvial_lag[5]['magnitude_of_slope'].max()

# Combined min and max for consistent binning
min_val = min(min_slope_alluvial, min_slope_non_alluvial)
max_val = max(max_slope_alluvial, max_slope_non_alluvial)

# Create bins with spacing of 0.1
bins = np.arange(min_val, max_val + 0.1, 0.1)

# Plot histograms
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True)

# Alluvial
axes[0].hist(alluvial_lag[5]['magnitude_of_slope'].dropna(), bins=bins, color='skyblue', edgecolor='black')
axes[0].set_title('Alluvial: Distribution of Magnitude of Slope')
axes[0].set_xlabel('Magnitude of Slope')
axes[0].set_ylabel('Frequency')

# Non-Alluvial
axes[1].hist(non_alluvial_lag[5]['magnitude_of_slope'].dropna(), bins=bins, color='salmon', edgecolor='black')
axes[1].set_title('Non-Alluvial: Distribution of Magnitude of Slope')
axes[1].set_xlabel('Magnitude of Slope')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# Define final covariate groups
target_col = 'magnitude_of_slope'
drop_cols = ['microwatershed_id', '.geo', 'lag', 'aquifer_category','sens_slope','mk_trend','mk_p_value','mk_tau','num_points','geometry','DeltaG','G']
numerical_cols = [
    'ET', 'Precipitation',
    'DeltaG','G', # added for checking
    'RunOff', 'weighted_avg_yeild',
    'soge_t_minus_1', 'krkrz_by_total_pct', 'krz_by_total_pct',
    'max_stream_order',
'mws_connectivity_indegree','infiltration_rate',
'summed_upstream_catchment_area',
'cropping_krkrz_by_total_pct',
'cropping_krz_by_total_pct','ewt_cm','grace_slope'
] + [f'Area%_SO_{i}' for i in range(1, 4)]
#categorical_cols = ['terrain_cluster']

#categorical_cols = ['terrain_cluster']

# Container for all splits
train_test_splits = {
    'alluvial': {},
    'non_alluvial': {}
}

# Helper function to split by microwatershed ID
def stratified_split(df, test_frac=0.2):
    unique_ids = df['microwatershed_id'].unique()
    train_ids, test_ids = train_test_split(unique_ids, test_size=test_frac, random_state=42)
    train_df = df[df['microwatershed_id'].isin(train_ids)].copy()
    test_df = df[df['microwatershed_id'].isin(test_ids)].copy()
    return train_df, test_df

# Apply splitting to all seasons within each aquifer category
for lag_number, lag_df in alluvial_lag.items():
    train_df, test_df = stratified_split(lag_df)
    train_test_splits['alluvial'][lag_number] = {
        'train': train_df,
        'test': test_df,
        'train_rows': len(train_df),
        'test_rows': len(test_df)
    }

for lag_number, lag_df in non_alluvial_lag.items():
    train_df, test_df = stratified_split(lag_df)
    train_test_splits['non_alluvial'][lag_number] = {
        'train': train_df,
        'test': test_df,
        'train_rows': len(train_df),
        'test_rows': len(test_df)
    }

In [ ]:
covariate_cols = [col for col in train_df.columns if col not in ['microwatershed_id','year','start_year','end_year', '.geo', 'lag', 'aquifer_category','sens_slope','mk_trend','mk_p_value','mk_tau','num_points','geometry','DeltaG','G','Area%_SO_4',
 'Area%_SO_5',
 'Area%_SO_6',
 'Area%_SO_7',
 'Area%_SO_8',
 'Area%_SO_9',
 'Area%_SO_10',
 'Area%_SO_11',
 'CWD_Calculated',
  'latitude',
 'longitude',
  'uid',]]

In [ ]:
covariate_cols

In [ ]:
import pandas as pd
import numpy as np
import pandas.api.types as ptypes

def summarize_covariates(train_df, test_df, covariate_cols):
    summary = []

    for col in covariate_cols:
        if not ptypes.is_numeric_dtype(train_df[col]):
            continue  # Skip non-numeric columns

        train_stats = train_df[col].describe()
        test_stats = test_df[col].describe()

        summary.append({
            'Covariate': col,
            'Train Mean': train_stats['mean'],
            'Test Mean': test_stats['mean'],
            'Train Std': train_stats['std'],
            'Test Std': test_stats['std'],
            'Train Min': train_stats['min'],
            'Test Min': test_stats['min'],
            'Train Max': train_stats['max'],
            'Test Max': test_stats['max']
        })

    return pd.DataFrame(summary)


In [ ]:
for aquifer in ['alluvial', 'non_alluvial']:
    for lag in train_test_splits[aquifer]:
        print(f"\n=== {aquifer.upper()} – LAG {lag} ===")
        train_df = train_test_splits[aquifer][lag]['train']
        test_df = train_test_splits[aquifer][lag]['test']
        summary_df = summarize_covariates(train_df, test_df, covariate_cols)
        display(summary_df)  # or print(summary_df.to_string(index=False))


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_covariate_distributions(train_df, test_df, covariate_cols, title_prefix=""):
    for col in covariate_cols:
        plt.figure(figsize=(8, 4))
        sns.kdeplot(train_df[col], label='Train', shade=True)
        sns.kdeplot(test_df[col], label='Test', shade=True)
        plt.title(f'{title_prefix} Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Density')
        plt.legend()
        plt.tight_layout()
        plt.show()


In [ ]:
for aquifer in ['alluvial', 'non_alluvial']:
    for lag in train_test_splits[aquifer]:
      plot_covariate_distributions(train_df, test_df, numerical_cols, title_prefix=f"{aquifer.upper()} LAG {lag}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_covariate_distributions(train_df, test_df, covariate_cols, title_prefix=""):
    for col in covariate_cols:
        plt.figure(figsize=(8, 4))
        sns.kdeplot(train_df[col], label='Train', shade=True)
        #sns.kdeplot(test_df[col], label='Test', shade=True)
        plt.title(f'{title_prefix} Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Density')
        plt.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
for aquifer in ['alluvial', 'non_alluvial']:
    for lag in train_test_splits[aquifer]:
      plot_covariate_distributions(train_df, test_df, numerical_cols, title_prefix=f"{aquifer.upper()} LAG {lag}")

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# Define the bins and labels
bins = [-np.inf, -1, -0.5, 0, 1, 2, np.inf]
bin_labels = [0, 1, 2, 3, 4, 5]  # optional numeric labels




def stratified_split_by_bins(df, target_col, id_col='microwatershed_id', test_frac=0.2):
    # Step 1: Bin the target variable
    df['bin'] = pd.cut(df[target_col], bins=bins, labels=bin_labels)

    # Step 2: Map bin to microwatershed ID by taking the mode bin for each ID
    id_bin_map = df.groupby(id_col)['bin'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])

    # Step 3: Stratified split of IDs by their bin category
    unique_ids = id_bin_map.index.to_numpy()
    bins_for_ids = id_bin_map.to_numpy()

    train_ids, test_ids = train_test_split(unique_ids, test_size=test_frac, stratify=bins_for_ids, random_state=42)

    # Step 4: Return the actual dataframes
    train_df = df[df[id_col].isin(train_ids)].drop(columns='bin')
    test_df = df[df[id_col].isin(test_ids)].drop(columns='bin')

    return train_df, test_df


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# Define the bins
bins = [-np.inf, -1.5, -1, -0.6, -0.4, -0.2,  -0.1,  0, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8,1, np.inf]

def stratified_split_by_bins(df, target_col, id_col='microwatershed_id', test_frac=0.2):
    # Step 1: Bin the target variable and extract bin indices
    df['bin'] = pd.cut(df[target_col], bins=bins)
    df['bin_index'] = df['bin'].cat.codes  # Convert to numeric bin indices

    # 🔍 Print number of samples in each bin
    bin_counts = df['bin'].value_counts().sort_index()
    print("\n📊 Sample counts per bin:")
    for b, count in zip(bin_counts.index, bin_counts.values):
        print(f"  Bin {b}: {count} samples")

    # Step 2: Map bin to microwatershed ID by taking the mode bin for each ID
    id_bin_map = df.groupby(id_col)['bin_index'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])

    # Step 3: Stratified split of IDs by their bin category
    unique_ids = id_bin_map.index.to_numpy()
    bins_for_ids = id_bin_map.to_numpy()

    train_ids, test_ids = train_test_split(unique_ids, test_size=test_frac, stratify=bins_for_ids, random_state=42)

    # Step 4: Return the actual dataframes
    train_df = df[df[id_col].isin(train_ids)].drop(columns=['bin', 'bin_index'])
    test_df = df[df[id_col].isin(test_ids)].drop(columns=['bin', 'bin_index'])

    return train_df, test_df



In [ ]:
train_test_splits = {
    'alluvial': {},
    'non_alluvial': {}
}

for lag_number, lag_df in alluvial_lag.items():
    train_df, test_df = stratified_split_by_bins(lag_df.copy(), target_col=target_col)
    train_test_splits['alluvial'][lag_number] = {
        'train': train_df,
        'test': test_df,
        'train_rows': len(train_df),
        'test_rows': len(test_df)
    }

for lag_number, lag_df in non_alluvial_lag.items():
    train_df, test_df = stratified_split_by_bins(lag_df.copy(), target_col=target_col)
    train_test_splits['non_alluvial'][lag_number] = {
        'train': train_df,
        'test': test_df,
        'train_rows': len(train_df),
        'test_rows': len(test_df)
    }

In [ ]:
train_test_splits

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import math

In [ ]:
def plot_boxplot_grid(df, columns, title_prefix, plots_per_row=5):
    total_plots = len(columns)
    n_rows = math.ceil(total_plots / plots_per_row)

    fig, axes = plt.subplots(n_rows, plots_per_row, figsize=(plots_per_row * 4, n_rows * 4))
    axes = axes.flatten()

    for i, col in enumerate(columns):
        sns.boxplot(y=df[col], ax=axes[i])
        axes[i].set_title(col)
        axes[i].set_xlabel("")  # Remove x-label
    # Turn off unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(title_prefix, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

# Define boxplot columns
boxplot_cols = numerical_cols + [target_col]

# Loop through splits and plot
for aquifer_group in ['alluvial', 'non_alluvial']:
    for lag_number, split_data in train_test_splits[aquifer_group].items():
        train_df = split_data['train']
        test_df = split_data['test']

        title_train = f"{aquifer_group.capitalize()} - lag {lag} - Train Set"
        title_test = f"{aquifer_group.capitalize()} - lag {lag} - Test Set"

        print(f"\n=== {title_train} ===")
        plot_boxplot_grid(train_df, boxplot_cols, title_train, plots_per_row=5)

        print(f"\n=== {title_test} ===")
        plot_boxplot_grid(test_df, boxplot_cols, title_test, plots_per_row=5)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import math

def plot_histogram_grid(df, columns, title_prefix, plots_per_row=5, bins=30):
    total_plots = len(columns)
    n_rows = math.ceil(total_plots / plots_per_row)

    fig, axes = plt.subplots(n_rows, plots_per_row, figsize=(plots_per_row * 4, n_rows * 4))
    axes = axes.flatten()

    for i, col in enumerate(columns):
        sns.histplot(df[col].dropna(), bins=bins, kde=True, ax=axes[i])
        axes[i].set_title(col)
        axes[i].set_xlabel("")
    # Turn off unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(title_prefix, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

# Define histogram columns
histogram_cols = numerical_cols + [target_col]

# Loop through splits and plot
for aquifer_group in ['alluvial', 'non_alluvial']:
    for lag_number, split_data in train_test_splits[aquifer_group].items():
        train_df = split_data['train']
        test_df = split_data['test']

        title_train = f"{aquifer_group.capitalize()} - Lag {lag_number} - Train Set"
        title_test = f"{aquifer_group.capitalize()} - Lag {lag_number} - Test Set"

        print(f"\n=== {title_train} ===")
        plot_histogram_grid(train_df, histogram_cols, title_train, plots_per_row=5)

        print(f"\n=== {title_test} ===")
        plot_histogram_grid(test_df, histogram_cols, title_test, plots_per_row=5)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import math

def plot_histogram_grid(df, columns, title_prefix, plots_per_row=5, bins=30):
    total_plots = len(columns)
    n_rows = math.ceil(total_plots / plots_per_row)

    fig, axes = plt.subplots(n_rows, plots_per_row, figsize=(plots_per_row * 4, n_rows * 4))
    axes = axes.flatten()

    for i, col in enumerate(columns):
        sns.histplot(df[col].dropna(), bins=bins, kde=True, ax=axes[i])
        axes[i].set_title(col)
        axes[i].set_xlabel("")
    # Turn off unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(title_prefix, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

# Define histogram columns
histogram_cols = numerical_cols + [target_col]

# Loop through splits and plot
for aquifer_group in ['alluvial', 'non_alluvial']:
    for lag_number, split_data in train_test_splits[aquifer_group].items():
        train_df = split_data['train']
        test_df = split_data['test']

        title_train = f"{aquifer_group.capitalize()} - Lag {lag_number} - Train Set"
        title_test = f"{aquifer_group.capitalize()} - Lag {lag_number} - Test Set"

        print(f"\n=== {title_train} ===")
        plot_histogram_grid(train_df, histogram_cols, title_train, plots_per_row=5)

        print(f"\n=== {title_test} ===")
        plot_histogram_grid(test_df, histogram_cols, title_test, plots_per_row=5)


In [ ]:
categorical_cols_to_plot = ['terrain_cluster']  # or ['terrain_cluster'] if not one-hot encoded

def plot_histogram_grid(df, num_cols, cat_cols, title_prefix, plots_per_row=5, bins=30):
    total_plots = len(num_cols + cat_cols)
    n_rows = math.ceil(total_plots / plots_per_row)

    fig, axes = plt.subplots(n_rows, plots_per_row, figsize=(plots_per_row * 4, n_rows * 4))
    axes = axes.flatten()

    idx = 0
    # Plot numerical histograms
    for col in num_cols:
        sns.histplot(df[col].dropna(), bins=bins, kde=True, ax=axes[idx])
        axes[idx].set_title(col)
        axes[idx].set_xlabel("")
        idx += 1

    # Plot categorical counts
    for col in cat_cols:
        sns.countplot(x=df[col], ax=axes[idx])
        axes[idx].set_title(f"{col} (Categorical)")
        axes[idx].set_xlabel("")
        idx += 1

    # Turn off unused subplots
    for j in range(idx, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(title_prefix, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

# Now call it with:
for aquifer_group in ['alluvial', 'non_alluvial']:
    for lag_number, split_data in train_test_splits[aquifer_group].items():
        train_df = split_data['train']
        test_df = split_data['test']

        title_train = f"{aquifer_group.capitalize()} - Lag {lag_number} - Train Set"
        title_test = f"{aquifer_group.capitalize()} - Lag {lag_number} - Test Set"

        print(f"\n=== {title_train} ===")
        plot_histogram_grid(train_df, numerical_cols + [target_col], categorical_cols_to_plot, title_train)

        print(f"\n=== {title_test} ===")
        plot_histogram_grid(test_df, numerical_cols + [target_col], categorical_cols_to_plot, title_test)


In [ ]:
# Construct a summary DataFrame
summary_data = []

for aquifer, lag_years in train_test_splits.items():
    for lag_year, info in lag_years.items():
        summary_data.append({
            'Aquifer': aquifer,
            'Lag': lag_year,
            'Train Rows': info['train_rows'],
            'Test Rows': info['test_rows']
        })

# Convert to DataFrame and display
summary_df = pd.DataFrame(summary_data)
print(summary_df)

In [ ]:
lag5_data = {
    aquifer: {lag: info}
    for aquifer, lag_years in train_test_splits.items()
    for lag, info in lag_years.items()
    if lag == 5
}

In [ ]:
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
# Loop through all train/test splits
for aquifer, seasons in train_test_splits.items():
    for season_label, data in seasons.items():
        print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

        train_df = data['train'].copy()
        test_df = data['test'].copy()

        # One-hot encode categorical columns
        # train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
        # test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

        # Align columns between train and test
        train_cols = train_df.columns
        test_df = test_df.reindex(columns=train_cols, fill_value=0)

        # Select features
        X_train = train_df[numerical_cols]
                           #+ [col for col in train_df.columns if 'terrain_cluster_' in col]].copy()
        y_train = train_df[target_col]

        X_test = test_df[numerical_cols]
                         #+ [col for col in test_df.columns if 'terrain_cluster_' in col]].copy()
        y_test = test_df[target_col]

        # Convert bools to int
        for df in [X_train, X_test]:
            for col in df.select_dtypes(include='bool').columns:
                df[col] = df[col].astype(int)

        # Add constant term
        X_train_ols = sm.add_constant(X_train)
        X_test_ols = sm.add_constant(X_test)

        # Align test to train
        for col in X_train_ols.columns:
            if col not in X_test_ols.columns:
                X_test_ols[col] = 0
        X_test_ols = X_test_ols[X_train_ols.columns]

        # Ensure numeric dtype
        X_train_ols = X_train_ols.astype(float)
        y_train = y_train.astype(float)
        X_test_ols = X_test_ols.astype(float)
        y_test = y_test.astype(float)
        # Fit OLS model
        model = sm.OLS(y_train, X_train_ols).fit()

        # Predict
        y_train_pred = model.predict(X_train_ols)
        y_test_pred = model.predict(X_test_ols)

        # Calculate MAPE
        train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
        test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

        # Print summary
        print(model.summary())
        print(f"Train MAPE: {train_mape:.4f}")
        print(f"Test MAPE:  {test_mape:.4f}")

        # Scatter plots
        plt.figure(figsize=(12, 5))

        # Train scatter
        plt.subplot(1, 2, 1)
        plt.scatter(y_train, y_train_pred, alpha=0.5)
        plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
        plt.xlabel('Actual magnitude of slope')
        plt.ylabel('Predicted magnitude of slope')
        plt.title(f'{aquifer} – {lag} – Train')

        # Test scatter
        plt.subplot(1, 2, 2)
        plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
        plt.xlabel('Actual magnitude of slope')
        plt.ylabel('Predicted magnitude of slope')
        plt.title(f'{aquifer} – {lag} – Test')

        plt.tight_layout()
        plt.show()

*Data normalization*

In [ ]:
def run_normalized_ols(train_test_splits, numerical_cols, target_col):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            #train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            #test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

            # Align columns between train and test
            train_cols = train_df.columns
            test_df = test_df.reindex(columns=train_cols, fill_value=0)

            # Select features
            X_train = train_df[numerical_cols]
            #+ [col for col in train_df.columns if 'terrain_cluster_' in col]].copy()
            y_train = train_df[target_col]

            X_test = test_df[numerical_cols]
            #+[col for col in test_df.columns if 'terrain_cluster_' in col]].copy()
            y_test = test_df[target_col]

            # Convert bools to int
            for df in [X_train, X_test]:
                for col in df.select_dtypes(include='bool').columns:
                    df[col] = df[col].astype(int)

            # Add constant term
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align test columns to train
            for col in X_train_ols.columns:
                if col not in X_test_ols.columns:
                    X_test_ols[col] = 0
            X_test_ols = X_test_ols[X_train_ols.columns]

            # Generate inverse sample weights
            sample_weights, bin_labels, bin_counts, bin_weights = generate_inverse_sample_weights(y_train)
            print(f"Bin counts: {bin_counts}")
            print(f"Bin weights: {bin_weights}")

            # Check for NaNs or infs
            if X_train_ols.isnull().any().any() or np.isinf(X_train_ols.values).any():
                print("Warning: NaN or inf detected in X_train_ols")
                X_train_ols = X_train_ols.replace([np.inf, -np.inf], np.nan).dropna(axis=0)
                y_train = y_train.loc[X_train_ols.index]
                sample_weights = sample_weights[X_train_ols.index]

            # Fit weighted least squares model
            model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit()

            # Predict
            y_train_pred = model.predict(X_train_ols)
            y_test_pred = model.predict(X_test_ols)

            # Evaluate
            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

            print(model.summary())
            print(f"Train MAPE: {train_mape:.4f}")
            print(f"Test MAPE:  {test_mape:.4f}")

            # Scatter plots
            plt.figure(figsize=(12, 5))

            # Train scatter
            plt.subplot(1, 2, 1)
            plt.scatter(y_train, y_train_pred, alpha=0.5)
            min_val = min(y_train.min(), y_train_pred.min())
            max_val = max(y_train.max(), y_train_pred.max())
            plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='45° line')
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train')
            plt.axis('equal')
            plt.legend()

            # Test scatter
            plt.subplot(1, 2, 2)
            plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
            min_val = min(y_test.min(), y_test_pred.min())
            max_val = max(y_test.max(), y_test_pred.max())
            plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='45° line')
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test')
            plt.axis('equal')
            plt.legend()

            plt.tight_layout()
            plt.show()


In [ ]:
import numpy as np
import pandas as pd

# Assume y_train is a NumPy array or pandas Series
y = y_train.copy()

# Step 1: Assign bins manually
bins = [-np.inf, -1.5, -1, -0.6, -0.4, -0.2,  -0.1,  0, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8,1, np.inf]
bin_labels = np.digitize(y, bins) - 1

# Step 2: Count samples per bin
bin_counts = np.bincount(bin_labels)

# Step 3: Compute smoothed inverse weights (log-based smoothing)
# Adding 1 inside log to avoid log(0), which is undefined
smoothed_bin_weights = 1.0 / ((bin_counts + 1))

# Optional normalization so weights sum to len(y)
#smoothed_bin_weights *= len(y) / smoothed_bin_weights.sum()

# Step 4: Assign weight to each sample
sample_weights = smoothed_bin_weights[bin_labels]


In [ ]:
print("Bin edges:", bins)
print("Bin counts:", bin_counts)
print("Bin weights:", sample_weights)

In [ ]:
import numpy as np
import pandas as pd

def generate_inverse_sample_weights(y, bins=[-np.inf, -1, 1, np.inf], normalize=True):
    """
    Generate inverse frequency-based sample weights from a continuous target variable.

    Parameters:
    - y: array-like, target values
    - bins: list of bin edges (default: [-inf, -1, 1, inf])
    - normalize: whether to normalize weights to sum to len(y)

    Returns:
    - sample_weights: numpy array of weights, same length as y
    - bin_labels: bin index assigned to each sample
    - bin_counts: number of samples in each bin
    - bin_weights: weight assigned to each bin
    """
    y = np.asarray(y)

    # Digitize y values into bins
    bin_labels = np.digitize(y, bins) - 1  # bin indices: 0, 1, 2, ...

    # Count samples in each bin
    bin_counts = np.bincount(bin_labels)

    bin_weights = 1.0 / bin_counts
    #     bin_weights *= len(y) / bin_weights.sum()

    # Assign weight to each sample
    sample_weights = bin_weights[bin_labels]

    return sample_weights, bin_labels, bin_counts, bin_weights

In [ ]:
print("Columns in X_train before normalization:", X_train.columns.tolist())

In [ ]:
sample_weights, bin_labels, bin_counts, bin_weights = generate_inverse_sample_weights(y_train)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
def run_normalized_ols(train_test_splits, numerical_cols, target_col):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            #train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            #test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

            # Align test columns with train
            test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

            # Identify terrain cluster columns
            #terrain_cols = [col for col in train_df.columns if 'terrain_cluster_' in col]

            # Prepare X and y
            X_train = train_df[numerical_cols]
                               #+ terrain_cols].copy()
            y_train = train_df[target_col]

            X_test = test_df[numerical_cols]
                             #+ terrain_cols].copy()
            y_test = test_df[target_col]

            # Normalize numeric features
            scaler = StandardScaler()
            X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

            # Add constant
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align test columns again
            for col in X_train_ols.columns:
                if col not in X_test_ols.columns:
                    X_test_ols[col] = 0
            X_test_ols = X_test_ols[X_train_ols.columns]

            # Generate inverse sample weights
            sample_weights, bin_labels, bin_counts, bin_weights = generate_inverse_sample_weights(y_train,bins)
            print(f"Bin counts: {bin_counts}")
            print(f"Bin weights: {bin_weights}")

            # Ensure float dtype before NaN/Inf check
            X_train_ols = X_train_ols.astype(float)
            X_test_ols = X_test_ols.astype(float)

            # Now the check will work
            if X_train_ols.isnull().any().any() or np.isinf(X_train_ols.to_numpy()).any():
                print("Warning: NaN or inf detected in X_train_ols")
                X_train_ols = X_train_ols.replace([np.inf, -np.inf], np.nan).dropna()
                y_train = y_train.loc[X_train_ols.index]
                sample_weights = sample_weights[X_train_ols.index]

            # Fit model
            model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit()

            # Predict
            y_train_pred = model.predict(X_train_ols)
            y_test_pred = model.predict(X_test_ols)

            # Evaluate
            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

            print(model.summary())
            print(f"Train MAPE: {train_mape:.4f}")
            print(f"Test MAPE:  {test_mape:.4f}")

            # Compute global min and max across both train and test actuals
            y_min = min(y_train.min(), y_test.min())
            y_max = max(y_train.max(), y_test.max())

            plt.figure(figsize=(12, 5))

            # --- Train plot ---
            plt.subplot(1, 2, 1)
            plt.scatter(y_train, y_train_pred, alpha=0.5)
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')  # Force 1:1 aspect ratio

            # --- Test plot ---
            plt.subplot(1, 2, 2)
            plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')

            plt.tight_layout()
            plt.show()

In [ ]:
run_normalized_ols(train_test_splits, numerical_cols, target_col)

In [ ]:
alluvial_bins = [-np.inf,  -1, -0.6, -0.4, -0.2, -0.1, 0, 0.1, 0.2, 0.4, 0.6, 0.8, 1, 1.5, 2, np.inf]
non_alluvial_bins = [-np.inf, -1, -0.6, -0.4, -0.2,-0.1, 0, 0.1, 0.2, 0.4, 0.6, 0.8, 1, 2, 3, 4, np.inf]

In [ ]:
# Assume y_train is a NumPy array or pandas Series
y_alluvial = alluvial_lag[5][target_col].copy()
y_non_alluvial = non_alluvial_lag[5][target_col].copy()



# Step 1: Assign bins manually
alluvial_bins = [-np.inf, -1.5, -1, -0.6, -0.4, -0.2,  -0.1,  0, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8,1, np.inf]
alluvial_bin_labels = np.digitize(y_alluvial, alluvial_bins) - 1

# Step 2: Count samples per bin
alluvial_bin_counts = np.bincount(alluvial_bin_labels)

# Step 3: Compute smoothed inverse weights (log-based smoothing)
# Adding 1 inside log to avoid log(0), which is undefined
alluvial_smoothed_bin_weights = 1.0 / ((alluvial_bin_counts + 1))

# Optional normalization so weights sum to len(y)
#smoothed_bin_weights *= len(y) / smoothed_bin_weights.sum()

# Step 4: Assign weight to each sample
alluvial_sample_weights = alluvial_smoothed_bin_weights[alluvial_bin_labels]

# Step 1: Assign bins manually
# non_alluvial_bins = [-np.inf, -1.5, -1, -0.6, -0.4, -0.2,  -0.1, -0.05,  -0.01, 0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1, 1.5, 2, 3, np.inf]
non_alluvial_bins = [-np.inf, -1.5, -1, 0, 1, 2, 3, np.inf]
non_alluvial_bin_labels = np.digitize(y_non_alluvial, non_alluvial_bins) - 1

# Step 2: Count samples per bin
non_alluvial_bin_counts = np.bincount(non_alluvial_bin_labels)

# Step 3: Compute smoothed inverse weights (log-based smoothing)
# Adding 1 inside log to avoid log(0), which is undefined
non_alluvial_smoothed_bin_weights = 1.0 / ((non_alluvial_bin_counts + 1))

# Optional normalization so weights sum to len(y)
#smoothed_bin_weights *= len(y) / smoothed_bin_weights.sum()

# Step 4: Assign weight to each sample
non_alluvial_sample_weights = non_alluvial_smoothed_bin_weights[non_alluvial_bin_labels]

In [ ]:
print("Bin edges:",alluvial_bins)
print("Bin counts:",alluvial_bin_counts)
print("Bin weights:",alluvial_sample_weights)

print("Bin edges:",non_alluvial_bins)
print("Bin counts:",non_alluvial_bin_counts)
print("Bin weights:",non_alluvial_sample_weights)

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib
import os
from sklearn.preprocessing import MinMaxScaler

# Path to save models
#model_save_path = "[private path removed]


def run_ols_removing_columns(train_test_splits, numerical_cols, target_col):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            #train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            #test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

            # Align test columns with train
            test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

            # Identify terrain cluster columns
            #terrain_cols = [col for col in train_df.columns if 'terrain_cluster_' in col]

            # Prepare X and y
            X_train = train_df[numerical_cols].copy()
            y_train = train_df[target_col]

            X_test = test_df[numerical_cols].copy()
            y_test = test_df[target_col]


            # Normalize numeric features with MinMaxScaler (scale to [0, 1])
            scaler = MinMaxScaler()
            X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])


            # # 🚨 Drop selected features
            # cols_to_drop_alluvial = ["Precipitation",
            #                 "ET",
            #                 #"DeltaG",
            #                 "G",
            #                 #"RunOff",
            #                 "weighted_avg_yeild",
            #                 "soge_t_minus_1",
            #                 #"krkrz_by_total_pct",
            #                 #"krz_by_total_pct",
            #                 #"max_stream_order",
            #                 #"mws_connectivity_indegree",
            #                 "infiltration_rate",
            #                 "summed_upstream_catchment_area",
            #                 "cropping_krkrz_by_total_pct",
            #                 #"cropping_krz_by_total_pct",
            #                 "ewt_cm",
            #                 #"grace_slope",
            #                          ]  # example


            # # 🚨 Drop selected features
            # cols_to_drop_non_alluvial = ["Precipitation",
            #                 "ET",
            #                 #"DeltaG",
            #                 "G",
            #                 "RunOff",
            #                 "weighted_avg_yeild",
            #                 "soge_t_minus_1",
            #                 #"krkrz_by_total_pct",
            #                 #"krz_by_total_pct",
            #                 #"max_stream_order",
            #                 #"mws_connectivity_indegree",
            #                 #"infiltration_rate",
            #                 "summed_upstream_catchment_area",
            #                 "cropping_krkrz_by_total_pct",
            #                 #"cropping_krz_by_total_pct",
            #                 "ewt_cm",
            #                 #"grace_slope"
            #                              ]  # example


            cols_to_drop_alluvial = [#"Precipitation",
                            # "ET",
                            "DeltaG",
                            #"G",
                            "RunOff",
                            "weighted_avg_yeild",
                            "soge_t_minus_1",
                            # "krkrz_by_total_pct",
                            #"krz_by_total_pct",
                            # "max_stream_order",
                            #"mws_connectivity_indegree",
                            #"infiltration_rate",
                            #"summed_upstream_catchment_area",
                            # "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                            # "ewt_cm",
                            # "grace_slope",
                                     ]  # example


            cols_to_drop_non_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            "G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                            "krkrz_by_total_pct",
                            #"krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            "infiltration_rate",
                            #"summed_upstream_catchment_area",
                            "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                            "ewt_cm",
                            "grace_slope"
                                         ]  # example



            if aquifer == 'alluvial':
                cols_to_drop = cols_to_drop_alluvial
            else:
                cols_to_drop = cols_to_drop_non_alluvial
            X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
            X_test = X_test.drop(columns=cols_to_drop, errors="ignore")


            # Normalize numeric features
            #scaler = StandardScaler()
            #X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            #X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

            # Add constant
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align test columns again
            for col in X_train_ols.columns:
                if col not in X_test_ols.columns:
                    X_test_ols[col] = 0
            X_test_ols = X_test_ols[X_train_ols.columns]

            # Generate inverse sample weights
            if aquifer == 'alluvial':
                bins = alluvial_bins
            else:
                bins = non_alluvial_bins

            sample_weights, bin_labels, bin_counts, bin_weights = generate_inverse_sample_weights(y_train,bins)
            print(f"Bin counts: {bin_counts}")
            print(f"Bin weights: {bin_weights}")

            # Ensure float dtype before NaN/Inf check
            X_train_ols = X_train_ols.astype(float)
            X_test_ols = X_test_ols.astype(float)

            # Now the check will work
            if X_train_ols.isnull().any().any() or np.isinf(X_train_ols.to_numpy()).any():
                print("Warning: NaN or inf detected in X_train_ols")
                X_train_ols = X_train_ols.replace([np.inf, -np.inf], np.nan).dropna()
                y_train = y_train.loc[X_train_ols.index]
                sample_weights = sample_weights[X_train_ols.index]

            # Fit model
            model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit(cov_type='HC3')

            #model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit()


            # Save model
            # model_filename = f"{aquifer}_{season_label}_ols_model.pkl"
            # joblib.dump(model, os.path.join(model_save_path, model_filename))
            # print(f"Model saved at: {os.path.join(model_save_path, model_filename)}")


            # Predict
            y_train_pred = model.predict(X_train_ols)
            y_test_pred = model.predict(X_test_ols)

            # Evaluate
            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

            print(model.summary())
            print(f"Train MAPE: {train_mape:.4f}")
            print(f"Test MAPE:  {test_mape:.4f}")

            # Compute global min and max across both train and test actuals
            y_min = min(y_train.min(), y_test.min())
            y_max = max(y_train.max(), y_test.max())


            test_bin_labels = pd.cut(y_test, bins=bins, labels=False, include_lowest=True)



            plt.figure(figsize=(12, 5))

            # --- Train plot ---
            plt.subplot(1, 2, 1)
            plt.scatter(y_train, y_train_pred, alpha=0.5)
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')  # Force 1:1 aspect ratio

            # --- Test plot ---
            plt.subplot(1, 2, 2)
            plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')

            plt.tight_layout()
            plt.show()


            # --- Train plot with bins ---
            #plt.subplot(1, 2, 1)
            for bin_val in np.unique(bin_labels):
                mask = bin_labels == bin_val
                plt.scatter(y_train[mask], y_train_pred[mask], alpha=0.5, label=f"Bin {bin_val}")
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train (by bins)')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')
            plt.legend()


            plt.tight_layout()
            plt.show()



            # --- Train plot with bins ---
            #plt.subplot(1, 2, 1)
            for test_bin_val in np.unique(test_bin_labels):
                test_mask = test_bin_labels == test_bin_val
                plt.scatter(y_test[test_mask], y_test_pred[test_mask], alpha=0.5, label=f"Bin {test_bin_val}")
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test (by bins)')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')
            plt.legend()

            unique_bins = np.unique(bin_labels)
            fig, axes = plt.subplots(1, len(unique_bins), figsize=(15, 5), sharex=True, sharey=True)






            plt.tight_layout()
            plt.show()

In [ ]:
run_ols_removing_columns(train_test_splits, numerical_cols, target_col)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

# Example usage on your training design matrix
for aquifer, seasons in train_test_splits.items():
    for season_label, data in seasons.items():
        print(f"\n===== OLS Model: {aquifer} – {season_label} =====")
        X_train = data['train'][numerical_cols].copy()
        # 🚨 Drop selected features
        cols_to_drop_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            #"G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                             "krkrz_by_total_pct",
                            "krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            "infiltration_rate",
                            #"summed_upstream_catchment_area",
                             "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                             #"ewt_cm",
                            "grace_slope",
                                     ]  # example


        # 🚨 Drop selected features
        cols_to_drop_non_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            "G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                            "krkrz_by_total_pct",
                            #"krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            "infiltration_rate",
                            #"summed_upstream_catchment_area",
                            "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                            #"ewt_cm",
                            "grace_slope"
                                         ]  # example

        if aquifer == 'alluvial':
          cols_to_drop = cols_to_drop_alluvial
        else:
          cols_to_drop = cols_to_drop_non_alluvial
        X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
        X_train_ols = sm.add_constant(X_train)
        print(X_train_ols.columns)
        X_train_ols_no_const = X_train_ols.drop(columns="const", errors="ignore")
        vif_df = calculate_vif(X_train_ols_no_const)
        print(vif_df)

In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib
import os
from sklearn.preprocessing import MinMaxScaler

# Path to save models
#model_save_path = "[private path removed]


def run_ols_removing_columns(train_test_splits, numerical_cols, target_col):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            #train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            #test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

            # Align test columns with train
            test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

            # Identify terrain cluster columns
            #terrain_cols = [col for col in train_df.columns if 'terrain_cluster_' in col]

            # Prepare X and y
            X_train = train_df[numerical_cols].copy()
            y_train = train_df[target_col]

            X_test = test_df[numerical_cols].copy()
            y_test = test_df[target_col]


            # Normalize numeric features with MinMaxScaler (scale to [0, 1])
            scaler = MinMaxScaler()
            X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])


            # # 🚨 Drop selected features
            # cols_to_drop_alluvial = ["Precipitation",
            #                 "ET",
            #                 #"DeltaG",
            #                 "G",
            #                 #"RunOff",
            #                 "weighted_avg_yeild",
            #                 "soge_t_minus_1",
            #                 #"krkrz_by_total_pct",
            #                 #"krz_by_total_pct",
            #                 #"max_stream_order",
            #                 #"mws_connectivity_indegree",
            #                 "infiltration_rate",
            #                 "summed_upstream_catchment_area",
            #                 "cropping_krkrz_by_total_pct",
            #                 #"cropping_krz_by_total_pct",
            #                 "ewt_cm",
            #                 #"grace_slope",
            #                          ]  # example


            # # 🚨 Drop selected features
            # cols_to_drop_non_alluvial = ["Precipitation",
            #                 "ET",
            #                 #"DeltaG",
            #                 "G",
            #                 "RunOff",
            #                 "weighted_avg_yeild",
            #                 "soge_t_minus_1",
            #                 #"krkrz_by_total_pct",
            #                 #"krz_by_total_pct",
            #                 #"max_stream_order",
            #                 #"mws_connectivity_indegree",
            #                 #"infiltration_rate",
            #                 "summed_upstream_catchment_area",
            #                 "cropping_krkrz_by_total_pct",
            #                 #"cropping_krz_by_total_pct",
            #                 "ewt_cm",
            #                 #"grace_slope"
            #                              ]  # example


            cols_to_drop_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            #"G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                             "krkrz_by_total_pct",
                            "krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            "infiltration_rate",
                            #"summed_upstream_catchment_area",
                             "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                             #"ewt_cm",
                            "grace_slope",
                                     ]  # example


          # 🚨 Drop selected features
            cols_to_drop_non_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            "G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                            "krkrz_by_total_pct",
                            #"krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            "infiltration_rate",
                            #"summed_upstream_catchment_area",
                            "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                            #"ewt_cm",
                            "grace_slope"
                                         ]  # example



            if aquifer == 'alluvial':
                cols_to_drop = cols_to_drop_alluvial
            else:
                cols_to_drop = cols_to_drop_non_alluvial
            X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
            X_test = X_test.drop(columns=cols_to_drop, errors="ignore")


            # Normalize numeric features
            #scaler = StandardScaler()
            #X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            #X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

            # Add constant
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align test columns again
            for col in X_train_ols.columns:
                if col not in X_test_ols.columns:
                    X_test_ols[col] = 0
            X_test_ols = X_test_ols[X_train_ols.columns]

            # Generate inverse sample weights
            if aquifer == 'alluvial':
                bins = alluvial_bins
            else:
                bins = non_alluvial_bins

            sample_weights, bin_labels, bin_counts, bin_weights = generate_inverse_sample_weights(y_train,bins)
            print(f"Bin counts: {bin_counts}")
            print(f"Bin weights: {bin_weights}")

            # Ensure float dtype before NaN/Inf check
            X_train_ols = X_train_ols.astype(float)
            X_test_ols = X_test_ols.astype(float)

            # Now the check will work
            if X_train_ols.isnull().any().any() or np.isinf(X_train_ols.to_numpy()).any():
                print("Warning: NaN or inf detected in X_train_ols")
                X_train_ols = X_train_ols.replace([np.inf, -np.inf], np.nan).dropna()
                y_train = y_train.loc[X_train_ols.index]
                sample_weights = sample_weights[X_train_ols.index]

            # Fit model
            model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit(cov_type='HC3')

            #model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit()


            # Save model
            # model_filename = f"{aquifer}_{season_label}_ols_model.pkl"
            # joblib.dump(model, os.path.join(model_save_path, model_filename))
            # print(f"Model saved at: {os.path.join(model_save_path, model_filename)}")


            # Predict
            y_train_pred = model.predict(X_train_ols)
            y_test_pred = model.predict(X_test_ols)

            # Evaluate
            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

            print(model.summary())
            print(f"Train MAPE: {train_mape:.4f}")
            print(f"Test MAPE:  {test_mape:.4f}")

            # Compute global min and max across both train and test actuals
            y_min = min(y_train.min(), y_test.min())
            y_max = max(y_train.max(), y_test.max())


            test_bin_labels = pd.cut(y_test, bins=bins, labels=False, include_lowest=True)



            plt.figure(figsize=(12, 5))

            # --- Train plot ---
            plt.subplot(1, 2, 1)
            plt.scatter(y_train, y_train_pred, alpha=0.5)
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')  # Force 1:1 aspect ratio

            # --- Test plot ---
            plt.subplot(1, 2, 2)
            plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')

            plt.tight_layout()
            plt.show()


            # --- Train plot with bins ---
            #plt.subplot(1, 2, 1)
            for bin_val in np.unique(bin_labels):
                mask = bin_labels == bin_val
                plt.scatter(y_train[mask], y_train_pred[mask], alpha=0.5, label=f"Bin {bin_val}")
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Train (by bins)')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')
            plt.legend()


            plt.tight_layout()
            plt.show()



            # --- Train plot with bins ---
            #plt.subplot(1, 2, 1)
            for test_bin_val in np.unique(test_bin_labels):
                test_mask = test_bin_labels == test_bin_val
                plt.scatter(y_test[test_mask], y_test_pred[test_mask], alpha=0.5, label=f"Bin {test_bin_val}")
            plt.plot([y_min, y_max], [y_min, y_max], 'r--')  # 45-degree line
            plt.xlabel('Actual')
            plt.ylabel('Predicted')
            plt.title(f'{aquifer} – {season_label} – Test (by bins)')
            plt.xlim(y_min, y_max)
            plt.ylim(y_min, y_max)
            plt.gca().set_aspect('equal', adjustable='box')
            plt.legend()

            unique_bins = np.unique(bin_labels)
            fig, axes = plt.subplots(1, len(unique_bins), figsize=(15, 5), sharex=True, sharey=True)






            plt.tight_layout()
            plt.show()

In [ ]:
run_ols_removing_columns(train_test_splits, numerical_cols, target_col)

In [ ]:
import numpy as np
import pandas as pd

def generate_sqrt_sample_weights(y, bins=[-np.inf, -1, 1, np.inf], normalize=True):
    """
    Generate sample weights using square-root inverse frequency scaling.

    Parameters:
    - y: array-like, target values
    - bins: list of bin edges for discretizing y
    - normalize: whether to normalize weights to sum to len(y)

    Returns:
    - sample_weights: numpy array of weights, same length as y
    - bin_labels: bin index assigned to each sample
    - bin_counts: number of samples in each bin
    - bin_weights: weight assigned to each bin
    """
    y = np.asarray(y)

    # Digitize y into bins
    bin_labels = np.digitize(y, bins) - 1

    # Count samples in each bin
    bin_counts = np.bincount(bin_labels)

    # Square-root inverse weighting
    bin_weights = 1.0 / np.sqrt(bin_counts)

    # if normalize:
    #     bin_weights *= len(y) / bin_weights.sum()

    # Assign weight to each sample
    sample_weights = bin_weights[bin_labels]

    return sample_weights, bin_labels, bin_counts, bin_weights


In [ ]:
def generate_log_sample_weights(y, bins=[-np.inf, -1, 1, np.inf], normalize=True):
    """
    Generate sample weights using logarithmic inverse frequency scaling.

    Parameters:
    - y: array-like, target values
    - bins: list of bin edges for discretizing y
    - normalize: whether to normalize weights to sum to len(y)

    Returns:
    - sample_weights: numpy array of weights, same length as y
    - bin_labels: bin index assigned to each sample
    - bin_counts: number of samples in each bin
    - bin_weights: weight assigned to each bin
    """
    y = np.asarray(y)

    # Digitize y into bins
    bin_labels = np.digitize(y, bins) - 1

    # Count samples in each bin
    bin_counts = np.bincount(bin_labels)

    # Logarithmic inverse weighting
    bin_weights = 1.0 / np.log1p(bin_counts)  # log(1+count)

    # if normalize:
    #     bin_weights *= len(y) / bin_weights.sum()

    # Assign weight to each sample
    sample_weights = bin_weights[bin_labels]

    return sample_weights, bin_labels, bin_counts, bin_weights


In [ ]:
def run_ols_with_multiple_weightings(train_test_splits, numerical_cols, target_col, weight_methods=["inverse", "sqrt", "log"]):
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Model: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # Prepare X and y
            X_train = train_df[numerical_cols].copy()
            y_train = train_df[target_col]
            X_test = test_df[numerical_cols].copy()
            y_test = test_df[target_col]

            # Normalize features
            #scaler = MinMaxScaler()
            #X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            #X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])


            cols_to_drop_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            #"G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                             "krkrz_by_total_pct",
                            "krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            "infiltration_rate",
                            #"summed_upstream_catchment_area",
                             "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                             #"ewt_cm",
                            "grace_slope",
                                     ]  # example


          # 🚨 Drop selected features
            cols_to_drop_non_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            "G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                            "krkrz_by_total_pct",
                            #"krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            "infiltration_rate",
                            #"summed_upstream_catchment_area",
                            "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                            #"ewt_cm",
                            "grace_slope"
                                         ]  # example



            if aquifer == 'alluvial':
                cols_to_drop = cols_to_drop_alluvial
            else:
                cols_to_drop = cols_to_drop_non_alluvial
            X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
            X_test = X_test.drop(columns=cols_to_drop, errors="ignore")


            # Add constant
            X_train_ols = sm.add_constant(X_train)
            X_test_ols = sm.add_constant(X_test)

            # Align
            X_test_ols = X_test_ols.reindex(columns=X_train_ols.columns, fill_value=0)

            # Pick bins
            bins = alluvial_bins if aquifer == "alluvial" else non_alluvial_bins

            # Loop through each weighting method
            for method in weight_methods:
                if method == "inverse":
                    sample_weights, bin_labels, bin_counts, bin_weights = generate_inverse_sample_weights(y_train, bins)
                elif method == "sqrt":
                    sample_weights, bin_labels, bin_counts, bin_weights = generate_sqrt_sample_weights(y_train, bins)
                elif method == "log":
                    sample_weights, bin_labels, bin_counts, bin_weights = generate_log_sample_weights(y_train, bins)

                print(f"\n--- Using {method.upper()} weighting ---")
                print(f"Bin counts: {bin_counts}")
                print(f"Bin weights: {bin_weights}")

                # Fit model
                model = sm.WLS(y_train, X_train_ols, weights=sample_weights).fit(cov_type='HC3')

                # Predict
                y_train_pred = model.predict(X_train_ols)
                y_test_pred = model.predict(X_test_ols)

                # Evaluate
                train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
                test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

                print(model.summary())
                print(f"Train MAPE ({method}): {train_mape:.4f}")
                print(f"Test MAPE  ({method}): {test_mape:.4f}")

                # Scatter plots
                y_min = min(y_train.min(), y_test.min())
                y_max = max(y_train.max(), y_test.max())

                plt.figure(figsize=(12, 5))
                plt.subplot(1, 2, 1)
                plt.scatter(y_train, y_train_pred, alpha=0.5)
                plt.plot([y_min, y_max], [y_min, y_max], 'r--')
                plt.title(f"Train – {method.upper()}")
                plt.xlabel("Actual"); plt.ylabel("Predicted")
                plt.xlim(y_min, y_max); plt.ylim(y_min, y_max)
                plt.gca().set_aspect("equal")

                plt.subplot(1, 2, 2)
                plt.scatter(y_test, y_test_pred, alpha=0.5, color="orange")
                plt.plot([y_min, y_max], [y_min, y_max], 'r--')
                plt.title(f"Test – {method.upper()}")
                plt.xlabel("Actual"); plt.ylabel("Predicted")
                plt.xlim(y_min, y_max); plt.ylim(y_min, y_max)
                plt.gca().set_aspect("equal")

                plt.suptitle(f"{aquifer} – {season_label} – {method.upper()} weighting")
                plt.tight_layout()
                plt.show()


In [ ]:
run_ols_with_multiple_weightings(train_test_splits, numerical_cols, target_col)

In [ ]:
cols_to_drop_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            #"G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                             "krkrz_by_total_pct",
                            "krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            "infiltration_rate",
                            #"summed_upstream_catchment_area",
                             "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                             #"ewt_cm",
                            "grace_slope",
                                     ]  # example


# 🚨 Drop selected features
cols_to_drop_non_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            "G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                            "krkrz_by_total_pct",
                            #"krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            "infiltration_rate",
                            #"summed_upstream_catchment_area",
                            "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                            #"ewt_cm",
                            "grace_slope"
                                         ]  # example


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_percentage_error, confusion_matrix, classification_report
import statsmodels.api as sm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def run_ols_with_multiple_weightings_and_classifier(
    train_test_splits,
    numerical_cols,
    target_col,
    weight_methods=["inverse", "sqrt", "log"],
    threshold=0.5,                  # define "near zero" band: [-threshold, +threshold]
    scale_features=False,           # set True if you want MinMax scaling

):

    # Map name -> weighting function (must exist in your namespace)
    weight_func_map = {
        "inverse": generate_inverse_sample_weights,
        "sqrt":    generate_sqrt_sample_weights,
        "log":     generate_log_sample_weights
    }

    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== OLS Cascade: {aquifer} – {season_label} =====")

            train_df = data['train'].copy()
            test_df  = data['test'].copy()

            # Build X, y
            X_train = train_df[numerical_cols].copy()
            y_train = train_df[target_col].copy()
            X_test  = test_df[numerical_cols].copy()
            y_test  = test_df[target_col].copy()

            # Normalize features
            scaler = MinMaxScaler()
            X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
            X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

            # Optional per-aquifer drops
            drop_list = cols_to_drop_alluvial if aquifer == "alluvial" else cols_to_drop_non_alluvial
            X_train = X_train.drop(columns=drop_list, errors="ignore")
            X_test  = X_test.drop(columns=drop_list, errors="ignore")

            # Optional feature scaling
            if scale_features:
                scaler = MinMaxScaler()
                X_train.loc[:, :] = scaler.fit_transform(X_train)
                # reindex test to train cols before transform to be safe
                X_test = X_test.reindex(columns=X_train.columns, fill_value=0.0)
                X_test.loc[:, :]  = scaler.transform(X_test)
            else:
                X_test = X_test.reindex(columns=X_train.columns, fill_value=0.0)

            # =========================
            # 1) CLASSIFIER (near vs away)
            # =========================
            # 0 = near zero, 1 = away from zero
            y_class_train = (y_train.abs() > threshold).astype(int)

            if y_class_train.nunique() < 2:
                print("Warning: only one class in training for this split; "
                      "falling back to single-regressor evaluation per weighting method.")
                # Fallback: single WLS with chosen weighting (no routing).
                # We still loop over weight_methods for apples-to-apples comparison.
                Xtr_ols = sm.add_constant(X_train.astype(float))
                Xte_ols = sm.add_constant(X_test.astype(float))
                bins = alluvial_bins if aquifer == "alluvial" else non_alluvial_bins

                for method in weight_methods:
                    wf = weight_func_map[method]
                    sample_weights, bin_labels, bin_counts, bin_weights = wf(y_train, bins)
                    print(f"\n--- Using {method.upper()} weighting (single WLS fallback) ---")
                    print(f"Bin counts: {bin_counts}")
                    print(f"Bin weights: {bin_weights}")

                    model = sm.WLS(y_train, Xtr_ols, weights=sample_weights).fit(cov_type='HC3')
                    y_train_pred = model.predict(Xtr_ols)
                    y_test_pred  = model.predict(Xte_ols)

                    train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
                    test_mape  = mean_absolute_percentage_error(y_test,  y_test_pred)
                    print(model.summary())
                    print(f"Train MAPE ({method}): {train_mape:.4f}")
                    print(f"Test  MAPE ({method}): {test_mape:.4f}")

                    y_min = float(min(y_train.min(), y_test.min()))
                    y_max = float(max(y_train.max(), y_test.max()))
                    plt.figure(figsize=(12, 5))
                    plt.subplot(1, 2, 1)
                    plt.scatter(y_train, y_train_pred, alpha=0.5)
                    plt.plot([y_min, y_max], [y_min, y_max], 'r--')
                    plt.title(f"Train – {method.upper()} (fallback)")
                    plt.xlabel("Actual"); plt.ylabel("Predicted")
                    plt.xlim(y_min, y_max); plt.ylim(y_min, y_max); plt.gca().set_aspect("equal")

                    plt.subplot(1, 2, 2)
                    plt.scatter(y_test, y_test_pred, alpha=0.5, color="orange")
                    plt.plot([y_min, y_max], [y_min, y_max], 'r--')
                    plt.title(f"Test – {method.UPPER()} (fallback)")
                    plt.xlabel("Actual"); plt.ylabel("Predicted")
                    plt.xlim(y_min, y_max); plt.ylim(y_min, y_max); plt.gca().set_aspect("equal")

                    plt.suptitle(f"{aquifer} – {season_label} – {method.upper()} weighting")
                    plt.tight_layout()
                    plt.show()

                continue  # go to next split

            # Train classifier
            clf = LogisticRegression(class_weight="balanced", max_iter=1000)
            clf.fit(X_train, y_class_train)

            # Masks for regimes
            near_mask_tr = (y_class_train == 0)
            away_mask_tr = ~near_mask_tr

            # Precompute const design matrices (keep same cols for both regimes)
            Xtr_all = sm.add_constant(X_train.astype(float))
            Xte_all = sm.add_constant(X_test.astype(float))

            # Pick bins by aquifer
            bins = alluvial_bins if aquifer == "alluvial" else non_alluvial_bins

            # =========================
            # Loop per weighting method
            # =========================
            for method in weight_methods:
                print(f"\n--- Using {method.upper()} weighting with classifier routing ---")
                wf = weight_func_map[method]

                # Fit NEAR regressor (if enough samples)
                model_near = None
                if near_mask_tr.sum() >= (Xtr_all.shape[1] + 1):
                    ytr_near = y_train[near_mask_tr]
                    Xtr_near = Xtr_all.loc[near_mask_tr]
                    w_near, _, near_counts, near_w = wf(ytr_near, bins)
                    print(f" NEAR counts: {near_counts}")
                    print(f" NEAR weights: {near_w}")
                    model_near = sm.WLS(ytr_near, Xtr_near, weights=w_near).fit(cov_type='HC3')
                else:
                    print(" Note: not enough NEAR samples; skipping NEAR regressor.")

                # Fit AWAY regressor (if enough samples)
                model_away = None
                if away_mask_tr.sum() >= (Xtr_all.shape[1] + 1):
                    ytr_away = y_train[away_mask_tr]
                    Xtr_away = Xtr_all.loc[away_mask_tr]
                    w_away, _, away_counts, away_w = wf(ytr_away, bins)
                    print(f" AWAY counts: {away_counts}")
                    print(f" AWAY weights: {away_w}")
                    model_away = sm.WLS(ytr_away, Xtr_away, weights=w_away).fit(cov_type='HC3')
                else:
                    print(" Note: not enough AWAY samples; skipping AWAY regressor.")

                # Route predictions by classifier
                y_class_pred_tr = clf.predict(X_train)
                y_class_pred_te = clf.predict(X_test)

                y_train_pred = np.zeros_like(y_train, dtype=float)
                y_test_pred  = np.zeros_like(y_test,  dtype=float)

                if model_near is not None:
                    mask = (y_class_pred_tr == 0)
                    if mask.any():
                        y_train_pred[mask] = model_near.predict(Xtr_all.loc[mask])
                if model_away is not None:
                    mask = (y_class_pred_tr == 1)
                    if mask.any():
                        y_train_pred[mask] = model_away.predict(Xtr_all.loc[mask])

                if model_near is not None:
                    mask = (y_class_pred_te == 0)
                    if mask.any():
                        y_test_pred[mask] = model_near.predict(Xte_all.loc[mask])
                if model_away is not None:
                    mask = (y_class_pred_te == 1)
                    if mask.any():
                        y_test_pred[mask] = model_away.predict(Xte_all.loc[mask])

                # Classifier diagnostics on test set
                print("\nClassifier diagnostics (test):")
                true_cls_test = (y_test.abs() > threshold).astype(int)
                print(confusion_matrix(true_cls_test, y_class_pred_te))
                print(classification_report(true_cls_test, y_class_pred_te, digits=3))

                # Metrics
                train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
                test_mape  = mean_absolute_percentage_error(y_test,  y_test_pred)
                print(f"Train MAPE ({method}): {train_mape:.4f}")
                print(f"Test  MAPE ({method}): {test_mape:.4f}")

                # Plots
                y_min = float(min(y_train.min(), y_test.min()))
                y_max = float(max(y_train.max(), y_test.max()))

                # Fit NEAR regressor (if enough samples)
                model_near = None
                if near_mask_tr.sum() >= (Xtr_all.shape[1] + 1):
                    ...
                    model_near = sm.WLS(ytr_near, Xtr_near, weights=w_near).fit(cov_type='HC3')
                    print("\n--- OLS Summary (NEAR samples) ---")
                    print(model_near.summary())

                # Fit AWAY regressor (if enough samples)
                model_away = None
                if away_mask_tr.sum() >= (Xtr_all.shape[1] + 1):
                    ...
                    model_away = sm.WLS(ytr_away, Xtr_away, weights=w_away).fit(cov_type='HC3')
                    print("\n--- OLS Summary (AWAY samples) ---")
                    print(model_away.summary())

                plt.figure(figsize=(12, 5))
                plt.subplot(1, 2, 1)
                plt.scatter(y_train, y_train_pred, alpha=0.5)
                plt.plot([y_min, y_max], [y_min, y_max], 'r--')
                plt.title(f"Train – {method.upper()} (Classifier Cascade)")
                plt.xlabel("Actual"); plt.ylabel("Predicted")
                plt.xlim(y_min, y_max); plt.ylim(y_min, y_max)
                plt.gca().set_aspect("equal", "box")

                plt.subplot(1, 2, 2)
                plt.scatter(y_test, y_test_pred, alpha=0.5, color="orange")
                plt.plot([y_min, y_max], [y_min, y_max], 'r--')
                plt.title(f"Test – {method.upper()} (Classifier Cascade)")
                plt.xlabel("Actual"); plt.ylabel("Predicted")
                plt.xlim(y_min, y_max); plt.ylim(y_min, y_max)
                plt.gca().set_aspect("equal", "box")

                plt.suptitle(f"{aquifer} – {season_label} – {method.upper()} weighting")
                plt.tight_layout()
                plt.show()

In [ ]:
run_ols_with_multiple_weightings_and_classifier(train_test_splits, numerical_cols, target_col)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

# Example usage on your training design matrix
for aquifer, seasons in train_test_splits.items():
    for season_label, data in seasons.items():
        print(f"\n===== OLS Model: {aquifer} – {season_label} =====")
        X_train = data['train'][numerical_cols].copy()
        # 🚨 Drop selected features
        cols_to_drop_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            #"G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                            "krkrz_by_total_pct",
                            #"krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            #"infiltration_rate",
                            #"summed_upstream_catchment_area",
                            "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                            "ewt_cm",
                            "grace_slope",
                                     ]  # example


        # 🚨 Drop selected features
        cols_to_drop_non_alluvial = [#"Precipitation",
                            "ET",
                            #"DeltaG",
                            "G",
                            "RunOff",
                            "weighted_avg_yeild",
                            #"soge_t_minus_1",
                            "krkrz_by_total_pct",
                            #"krz_by_total_pct",
                            "max_stream_order",
                            #"mws_connectivity_indegree",
                            #"infiltration_rate",
                            #"summed_upstream_catchment_area",
                            "cropping_krkrz_by_total_pct",
                            #"cropping_krz_by_total_pct",
                            "ewt_cm",
                            "grace_slope"
                                         ]  # example

        if aquifer == 'alluvial':
          cols_to_drop = cols_to_drop_alluvial
        else:
          cols_to_drop = cols_to_drop_non_alluvial
        X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
        X_train_ols = sm.add_constant(X_train)
        print(X_train_ols.columns)
        X_train_ols_no_const = X_train_ols.drop(columns="const", errors="ignore")
        vif_df = calculate_vif(X_train_ols_no_const)
        print(vif_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

def plot_covariate_distributions(train_test_splits, numerical_cols, bins=30):
    """
    Plot histogram + KDE + boxplot + QQ-plot for each numerical covariate
    in the training data. Helps decide which normalization/transform to use.
    """
    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            print(f"\n===== Covariate Distributions: {aquifer} – {season_label} =====")

            X_train = data['train'][numerical_cols].copy()
            X_test  = data['test'][numerical_cols].copy()

            # 🚨 Drop selected features
            cols_to_drop_alluvial = [#"Precipitation",
                                "ET",
                                #"DeltaG",
                                #"G",
                                "RunOff",
                                "weighted_avg_yeild",
                                #"soge_t_minus_1",
                                "krkrz_by_total_pct",
                                #"krz_by_total_pct",
                                "max_stream_order",
                                #"mws_connectivity_indegree",
                                #"infiltration_rate",
                                #"summed_upstream_catchment_area",
                                "cropping_krkrz_by_total_pct",
                                #"cropping_krz_by_total_pct",
                                "ewt_cm",
                                "grace_slope",
                                          ]  # example


            # 🚨 Drop selected features
            cols_to_drop_non_alluvial = [#"Precipitation",
                                "ET",
                                #"DeltaG",
                                "G",
                                "RunOff",
                                "weighted_avg_yeild",
                                #"soge_t_minus_1",
                                "krkrz_by_total_pct",
                                #"krz_by_total_pct",
                                "max_stream_order",
                                #"mws_connectivity_indegree",
                                #"infiltration_rate",
                                #"summed_upstream_catchment_area",
                                "cropping_krkrz_by_total_pct",
                                #"cropping_krz_by_total_pct",
                                "ewt_cm",
                                "grace_slope"
                                              ]  # example

            cols_to_drop = cols_to_drop_alluvial if aquifer == 'alluvial' else cols_to_drop_non_alluvial
            X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
            X_test  = X_test.drop(columns=cols_to_drop, errors="ignore")

            # Loop over remaining numerical covariates
            for col in X_train.columns:
                fig, axes = plt.subplots(1, 3, figsize=(18, 4))

                # Histogram + KDE
                sns.histplot(X_train[col].dropna(), bins=bins, kde=True, ax=axes[0], color="skyblue")
                axes[0].set_title(f"{col} – Histogram + KDE")

                # Boxplot
                sns.boxplot(x=X_train[col].dropna(), ax=axes[1], color="orange")
                axes[1].set_title(f"{col} – Boxplot")

                # QQ-plot
                stats.probplot(X_train[col].dropna(), dist="norm", plot=axes[2])
                axes[2].set_title(f"{col} – QQ-plot")

                plt.suptitle(f"Distribution of {col} ({aquifer}, {season_label})", fontsize=14)
                plt.tight_layout()
                plt.show()


In [ ]:
plot_covariate_distributions(train_test_splits,numerical_cols)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer

def normalize_and_plot(df, normal_cols=None, right_skewed_cols=None, left_skewed_cols=None, heavy_tail_cols=None, bins=30):
    """
    Apply distribution-aware normalization and plot before/after histograms.
    df : DataFrame
    *_cols : lists of column names for each distribution type
    """

    normal_cols = normal_cols or []
    right_skewed_cols = right_skewed_cols or []
    left_skewed_cols = left_skewed_cols or []
    heavy_tail_cols = heavy_tail_cols or []

    df_norm = df.copy()

    # --- Normal (StandardScaler) ---
    scaler = StandardScaler()
    if normal_cols:
        df_norm[normal_cols] = scaler.fit_transform(df[normal_cols])

    # --- Right-skewed (log transform) ---
    for col in right_skewed_cols:
        # Add +1 shift if non-negative, else fallback to Yeo-Johnson
        if (df[col] >= 0).all():
            df_norm[col] = np.log1p(df[col])
        else:
            pt = PowerTransformer(method="yeo-johnson")
            df_norm[col] = pt.fit_transform(df[[col]])

    # --- Left-skewed (Yeo-Johnson handles negatives) ---
    for col in left_skewed_cols:
        pt = PowerTransformer(method="yeo-johnson")
        df_norm[col] = pt.fit_transform(df[[col]])

    # --- Heavy-tailed (RobustScaler) ---
    if heavy_tail_cols:
        rscaler = RobustScaler()
        df_norm[heavy_tail_cols] = rscaler.fit_transform(df[heavy_tail_cols])

    # --- Plot Before/After ---
    for col in normal_cols + right_skewed_cols + left_skewed_cols + heavy_tail_cols:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        sns.histplot(df[col].dropna(), bins=bins, kde=True, ax=axes[0], color="steelblue")
        axes[0].set_title(f"{col} – Before")

        sns.histplot(df_norm[col].dropna(), bins=bins, kde=True, ax=axes[1], color="seagreen")
        axes[1].set_title(f"{col} – After")

        plt.suptitle(f"Normalization of {col}")
        plt.tight_layout()
        plt.show()

    return df_norm

In [ ]:
for aquifer, seasons in train_test_splits.items():
    for season_label, data in seasons.items():
        print(f"\n===== Covariate Distributions: {aquifer} – {season_label} =====")

        X_train = data['train'][numerical_cols].copy()
        X_test  = data['test'][numerical_cols].copy()

        # 🚨 Drop selected features
        cols_to_drop_alluvial = [#"Precipitation",
                                "ET",
                                #"DeltaG",
                                "G",
                                "RunOff",
                                "weighted_avg_yeild",
                                #"soge_t_minus_1",
                                "krkrz_by_total_pct",
                                #"krz_by_total_pct",
                                "max_stream_order",
                                #"mws_connectivity_indegree",
                                #"infiltration_rate",
                                #"summed_upstream_catchment_area",
                                "cropping_krkrz_by_total_pct",
                                #"cropping_krz_by_total_pct",
                                "ewt_cm",
                                "grace_slope",
                                "Area%_SO_1",
                                "Area%_SO_2",
                                #"Area%_SO_3"
                                        ]  # example


            # 🚨 Drop selected features
        cols_to_drop_non_alluvial = [#"Precipitation",
                                "ET",
                                #"DeltaG",
                                "G",
                                "RunOff",
                                "weighted_avg_yeild",
                                #"soge_t_minus_1",
                                "krkrz_by_total_pct",
                                #"krz_by_total_pct",
                                "max_stream_order",
                                #"mws_connectivity_indegree",
                                "infiltration_rate",
                                #"summed_upstream_catchment_area",
                                "cropping_krkrz_by_total_pct",
                                #"cropping_krz_by_total_pct",
                                "ewt_cm",
                                "grace_slope",
                                "Area%_SO_1",
                                "Area%_SO_2",
                                #"Area%_SO_3"
                                ]


        # []

        # []



        cols_to_drop = cols_to_drop_alluvial if aquifer == 'alluvial' else cols_to_drop_non_alluvial
        X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
        X_test  = X_test.drop(columns=cols_to_drop, errors="ignore")
        if aquifer == 'alluvial':
          df_norm = normalize_and_plot(
              df=X_train,
              normal_cols=["DeltaG", "mws_connectivity_indegree",
                           ],
              right_skewed_cols=["infiltration_rate"],
              left_skewed_cols=["Precipitation","soge_t_minus_1","Area%_SO_3","krz_by_total_pct","summed_upstream_catchment_area","cropping_krz_by_total_pct"],
              heavy_tail_cols=[]
          )
        else:
          df_norm = normalize_and_plot(
              df=X_train,
              normal_cols=["Precipitation"],
              right_skewed_cols=["soge_t_minus_1","DeltaG"],
              left_skewed_cols=["Area%_SO_3","cropping_krz_by_total_pct","summed_upstream_catchment_area","mws_connectivity_indegree","krz_by_total_pct"],
              heavy_tail_cols=[]
          )

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer

# ---- 1) per-distribution normalizer (fit on train, transform train+test) ----
def normalize_by_distribution(X_train, X_test, cfg):
    """
    cfg = {
      'normal':        [...],  # z-score (StandardScaler)
      'right_skewed':  [...],  # log1p if >=0 else Yeo-Johnson
      'left_skewed':   [...],  # Yeo-Johnson
      'heavy_tail':    [...]   # RobustScaler
    }
    """
    Xtr = X_train.copy()
    Xte = X_test.copy()

    def cols_in(df, cols):  # keep only columns that exist after any dropping
        return [c for c in (cols or []) if c in df.columns]

    # Right-skewed: log1p if nonnegative; else Yeo–Johnson
    for col in cols_in(Xtr, cfg.get('right_skewed')):
        if (Xtr[col].min() >= 0) and (Xte[col].min() >= 0):
            Xtr[col] = np.log1p(Xtr[col])
            Xte[col] = np.log1p(Xte[col])
        else:
            pt = PowerTransformer(method="yeo-johnson")
            Xtr[col] = pt.fit_transform(Xtr[[col]]).ravel()
            Xte[col] = pt.transform(Xte[[col]]).ravel()

    # Left-skewed: Yeo–Johnson
    for col in cols_in(Xtr, cfg.get('left_skewed')):
        pt = PowerTransformer(method="yeo-johnson")
        Xtr[col] = pt.fit_transform(Xtr[[col]]).ravel()
        Xte[col] = pt.transform(Xte[[col]]).ravel()

    # Normal-ish: StandardScaler
    norm_cols = cols_in(Xtr, cfg.get('normal'))
    if norm_cols:
        ss = StandardScaler()
        Xtr[norm_cols] = ss.fit_transform(Xtr[norm_cols])
        Xte[norm_cols] = ss.transform(Xte[norm_cols])

    # Heavy-tailed / outliers: RobustScaler
    heavy_cols = cols_in(Xtr, cfg.get('heavy_tail'))
    if heavy_cols:
        rs = RobustScaler()
        Xtr[heavy_cols] = rs.fit_transform(Xtr[heavy_cols])
        Xte[heavy_cols] = rs.transform(Xte[heavy_cols])

    return Xtr, Xte

# ---- 2) build normalized splits dict that mirrors your original structure ----
def build_normalized_splits(train_test_splits, numerical_cols, target_col, cfg_by_aquifer,
                            cols_to_drop_alluvial=None, cols_to_drop_non_alluvial=None):
    cols_to_drop_alluvial = cols_to_drop_alluvial or []
    cols_to_drop_non_alluvial = cols_to_drop_non_alluvial or []

    splits_norm = {}
    for aquifer, seasons in train_test_splits.items():
        splits_norm[aquifer] = {}
        drop_list = cols_to_drop_alluvial if aquifer == "alluvial" else cols_to_drop_non_alluvial
        cfg = cfg_by_aquifer.get(aquifer, {})  # distribution config for this aquifer

        for season_label, data in seasons.items():
            train_df = data['train'].copy()
            test_df  = data['test'].copy()

            # select features then drop per-aquifer list (if present)
            Xtr = train_df[numerical_cols].drop(columns=drop_list, errors="ignore")
            Xte = test_df[numerical_cols].drop(columns=drop_list, errors="ignore")

            # normalize per distribution (fit on train, transform both)
            Xtr_norm, Xte_norm = normalize_by_distribution(Xtr, Xte, cfg)

            # ensure same column order/coverage
            Xte_norm = Xte_norm.reindex(columns=Xtr_norm.columns, fill_value=0.0)

            # reattach target
            train_norm = pd.concat([Xtr_norm, train_df[[target_col]]], axis=1)
            test_norm  = pd.concat([Xte_norm,  test_df[[target_col]]],  axis=1)

            splits_norm[aquifer][season_label] = {
                'train': train_norm,
                'test':  test_norm
            }

    return splits_norm

# ---- 3) example config: reuse what you were plotting with normalize_and_plot ----
norm_cfg = {
    'alluvial': {
        'normal':       ["DeltaG", "mws_connectivity_indegree"],
        'right_skewed': ["infiltration_rate"],
        'left_skewed':  ["Precipitation","soge_t_minus_1","Area%_SO_3",
                         "krz_by_total_pct","summed_upstream_catchment_area",
                         "cropping_krz_by_total_pct"],
        'heavy_tail':   []
    },
    'non_alluvial': {
        'normal':       ["Precipitation"],
        'right_skewed': ["soge_t_minus_1","DeltaG"],
        'left_skewed':  ["Area%_SO_3","cropping_krz_by_total_pct",
                         "summed_upstream_catchment_area","mws_connectivity_indegree",
                         "krz_by_total_pct"],
        'heavy_tail':   []
    }
}

# 🚨 Drop selected features
cols_to_drop_alluvial = [#"Precipitation",
                        "ET",
                        #"DeltaG",
                        "G",
                        "RunOff",
                        "weighted_avg_yeild",
                        #"soge_t_minus_1",
                        "krkrz_by_total_pct",
                        #"krz_by_total_pct",
                        "max_stream_order",
                        #"mws_connectivity_indegree",
                        #"infiltration_rate",
                        #"summed_upstream_catchment_area",
                        "cropping_krkrz_by_total_pct",
                        #"cropping_krz_by_total_pct",
                        "ewt_cm",
                        "grace_slope",
                        "Area%_SO_1",
                        "Area%_SO_2",
                        #"Area%_SO_3"
                                ]  # example


    # 🚨 Drop selected features
cols_to_drop_non_alluvial = [#"Precipitation",
                        "ET",
                        #"DeltaG",
                        "G",
                        "RunOff",
                        "weighted_avg_yeild",
                        #"soge_t_minus_1",
                        "krkrz_by_total_pct",
                        #"krz_by_total_pct",
                        "max_stream_order",
                        #"mws_connectivity_indegree",
                        "infiltration_rate",
                        #"summed_upstream_catchment_area",
                        "cropping_krkrz_by_total_pct",
                        #"cropping_krz_by_total_pct",
                        "ewt_cm",
                        "grace_slope",
                        "Area%_SO_1",
                        "Area%_SO_2",
                        #"Area%_SO_3"
                        ]


In [ ]:
# 1) create a normalized copy of your splits
splits_norm = build_normalized_splits(
    train_test_splits=train_test_splits,
    numerical_cols=numerical_cols,     # your original feature list
    target_col=target_col,
    cfg_by_aquifer=norm_cfg,
    cols_to_drop_alluvial=cols_to_drop_alluvial,
    cols_to_drop_non_alluvial=cols_to_drop_non_alluvial
)

# 2) call your existing runner per aquifer with the **actual** normalized cols
for aquifer in splits_norm.keys():
    # pick one season to read the list of feature columns (same across seasons here)
    first_season = next(iter(splits_norm[aquifer]))
    num_cols_norm = [c for c in splits_norm[aquifer][first_season]['train'].columns if c != target_col]

    # build a mini dict for this aquifer only and pass to your OLS function
    mini_splits = {aquifer: splits_norm[aquifer]}
    run_ols_with_multiple_weightings(
        train_test_splits=mini_splits,
        numerical_cols=num_cols_norm,     # normalized feature list for this aquifer
        target_col=target_col,
        weight_methods=["inverse","sqrt","log"]
    )


In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, confusion_matrix, ConfusionMatrixDisplay

def classify_and_regress(
    X_train, y_train, X_test, y_test,
    threshold=2,
    clf_C=1.0, clf_class_weight="balanced", clf_threshold=0.5
):
    # Ensure arrays
    y_train = np.asarray(y_train)
    y_test  = np.asarray(y_test)

    # === Define classes based on actual values ===
    y_class_train = (y_train > threshold).astype(int)
    y_class_test  = (y_test  > threshold).astype(int)


    # === Train classifier with tunable hyperparams ===
    clf = LogisticRegression(
        C=clf_C,
        class_weight="balanced",
        max_iter=500
    )
    clf.fit(X_train, y_class_train)

    # === Train regressors ===
    reg_low  = LinearRegression()  # for y <= threshold
    reg_high = LinearRegression()  # for y > threshold

    reg_low.fit(X_train[y_class_train == 0], y_train[y_class_train == 0])
    reg_high.fit(X_train[y_class_train == 1], y_train[y_class_train == 1])

    # === Predict class labels ===
    y_class_pred_train = clf.predict(X_train)
    y_class_pred_test  = clf.predict(X_test)

    # === Route predictions ===
    def route_predictions(X, y_class_pred):
        preds = np.zeros(len(X))
        for i, label in enumerate(y_class_pred):
            if label == 0:
                preds[i] = reg_low.predict(X[i].reshape(1, -1))[0]
            else:
                preds[i] = reg_high.predict(X[i].reshape(1, -1))[0]
        return preds

    y_train_pred = route_predictions(X_train, y_class_pred_train)
    y_test_pred  = route_predictions(X_test,  y_class_pred_test)

    # === Evaluate ===
    train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
    test_mape  = mean_absolute_percentage_error(y_test,  y_test_pred)
    print(f"Train MAPE: {train_mape:.4f}")
    print(f"Test  MAPE: {test_mape:.4f}")

    # === Confusion Matrix (Test) ===
    cm = confusion_matrix(y_class_test, y_class_pred_test)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=["<= threshold", "> threshold"])
    disp.plot(cmap="Blues")
    plt.title("Classifier Confusion Matrix (Test)")
    plt.show()

    # === Scatter plots ===
    def plot_scatter(y_true, y_pred, y_class_true, split="Train"):
        plt.figure(figsize=(6,6))
        for label, color in zip([0,1], ["blue","red"]):
            mask = (y_class_true == label)
            plt.scatter(y_true[mask], y_pred[mask], alpha=0.6, c=color, label=f"Class {label}")
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        plt.plot([min_val, max_val],[min_val,max_val],'k--')
        plt.xlabel("Actual"); plt.ylabel("Predicted")
        plt.title(f"Actual vs Predicted ({split} Set)")
        plt.legend()
        plt.gca().set_aspect("equal","box")
        plt.show()

    plot_scatter(y_train, y_train_pred, y_class_train, split="Train")
    plot_scatter(y_test,  y_test_pred,  y_class_test,  split="Test")

    return clf, reg_low, reg_high, y_train_pred, y_test_pred


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Extract train/test
        train_df = data['train']
        test_df  = data['test']

        # Feature columns = all except target
        X_train = train_df.drop(columns=[target_col]).values
        y_train = train_df[target_col].values
        X_test  = test_df.drop(columns=[target_col]).values
        y_test  = test_df[target_col].values

        clf, reg_near, reg_away, y_train_pred, y_test_pred = classify_and_regress(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            threshold=-0.5,
            clf_C=0.1,                # smaller = stronger regularization
            clf_class_weight="balanced",
            clf_threshold=0.4         # adjust decision threshold
        )



In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Training and test sets (already normalized!)
        train_df = data['train']
        test_df  = data['test']

        # Feature columns (exclude target)
        num_cols_norm = [c for c in train_df.columns if c != target_col]

        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values


        clf, reg_near, reg_away, y_train_pred, y_test_pred = classify_and_regress(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            threshold=-0.5,
            clf_C=0.3,                # smaller = stronger regularization
            clf_class_weight="balanced",
            clf_threshold=0.4         # adjust decision threshold
        )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, confusion_matrix, ConfusionMatrixDisplay

def classify_and_regress_compare(X_train, y_train, X_test, y_test, threshold=2):
    """
    Compare ideal routing (true labels) vs classifier routing (predicted labels).
    """

    # Ensure arrays
    y_train = np.asarray(y_train)
    y_test  = np.asarray(y_test)

    # Define classes (based on true actual values)
    y_class_train = (y_train > threshold).astype(int)
    y_class_test  = (y_test  > threshold).astype(int)

    # === Train classifier ===
    clf = LogisticRegression(class_weight="balanced", max_iter=1000)
    clf.fit(X_train, y_class_train)

    # === Train regressors ===
    reg_low  = LinearRegression()
    reg_high = LinearRegression()

    reg_low.fit(X_train[y_class_train == 0], y_train[y_class_train == 0])
    reg_high.fit(X_train[y_class_train == 1], y_train[y_class_train == 1])

    # === Helper: route predictions ===
    def route_predictions(X, labels):
        preds = np.zeros(len(X))
        for i, label in enumerate(labels):
            if label == 0:
                preds[i] = reg_low.predict(X[i].reshape(1, -1))[0]
            else:
                preds[i] = reg_high.predict(X[i].reshape(1, -1))[0]
        return preds

    # 1️⃣ Ideal routing (true labels)
    y_train_pred_true = route_predictions(X_train, y_class_train)
    y_test_pred_true  = route_predictions(X_test,  y_class_test)

    # 2️⃣ Classifier routing (predicted labels)
    y_class_pred_train = clf.predict(X_train)
    y_class_pred_test  = clf.predict(X_test)
    y_train_pred_clf = route_predictions(X_train, y_class_pred_train)
    y_test_pred_clf  = route_predictions(X_test,  y_class_pred_test)

    # === Evaluate ===
    print("\n--- Metrics ---")
    print("Train MAPE (ideal routing):      ", mean_absolute_percentage_error(y_train, y_train_pred_true))
    print("Test  MAPE (ideal routing):      ", mean_absolute_percentage_error(y_test,  y_test_pred_true))
    print("Train MAPE (classifier routing): ", mean_absolute_percentage_error(y_train, y_train_pred_clf))
    print("Test  MAPE (classifier routing): ", mean_absolute_percentage_error(y_test,  y_test_pred_clf))

    # === Confusion Matrix ===
    cm = confusion_matrix(y_class_test, y_class_pred_test)
    ConfusionMatrixDisplay(cm, display_labels=["≤ threshold", "> threshold"]).plot(cmap="Blues")
    plt.title("Classifier Confusion Matrix (Test)")
    plt.show()

    # === Scatter Plots ===
    def plot_scatter(y_true, y_pred, y_class, split, mode):
        plt.figure(figsize=(6,6))
        for label, color in zip([0,1], ["blue","red"]):
            mask = (y_class == label)
            plt.scatter(y_true[mask], y_pred[mask], alpha=0.6, c=color, label=f"Class {label}")
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        plt.plot([min_val, max_val],[min_val,max_val],'k--')
        plt.xlabel("Actual"); plt.ylabel("Predicted")
        plt.title(f"{split} Set – {mode}")
        plt.legend(); plt.gca().set_aspect("equal","box")
        plt.show()

    # Train/Test for both routing types
    plot_scatter(y_train, y_train_pred_true, y_class_train, "Train", "Ideal Routing")
    plot_scatter(y_test,  y_test_pred_true,  y_class_test,  "Test",  "Ideal Routing")
    plot_scatter(y_train, y_train_pred_clf,  y_class_train, "Train", "Classifier Routing")
    plot_scatter(y_test,  y_test_pred_clf,   y_class_test,  "Test",  "Classifier Routing")

    return clf, reg_low, reg_high, y_train_pred_true, y_test_pred_true, y_train_pred_clf, y_test_pred_clf


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Training and test sets (already normalized!)
        train_df = data['train']
        test_df  = data['test']

        # Feature columns (exclude target)
        num_cols_norm = [c for c in train_df.columns if c != target_col]

        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values


        clf, reg_low, reg_high, y_train_pred_true, y_test_pred_true, y_train_pred_clf, y_test_pred_clf = classify_and_regress_compare(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            threshold=-0.5,
            #clf_C=0.3,                # smaller = stronger regularization
            #clf_class_weight="balanced",
            #clf_threshold=0.4         # adjust decision threshold
        )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, confusion_matrix, ConfusionMatrixDisplay

def classify_and_regress_compare(X_train, y_train, X_test, y_test, threshold=2):
    """
    Compare ideal routing (true labels) vs classifier routing (predicted labels).
    """

    # Ensure arrays
    y_train = np.asarray(y_train)
    y_test  = np.asarray(y_test)

    # Define classes (based on true actual values)
    y_class_train = (y_train > threshold).astype(int)
    y_class_test  = (y_test  > threshold).astype(int)

    # === Train classifier ===
    clf = LogisticRegression(class_weight="balanced", max_iter=1000)
    clf.fit(X_train, y_class_train)

    # === Train regressors ===
    reg_low  = LinearRegression()
    reg_high = LinearRegression()

    reg_low.fit(X_train[y_class_train == 0], y_train[y_class_train == 0])
    reg_high.fit(X_train[y_class_train == 1], y_train[y_class_train == 1])

    # === Helper: route predictions ===
    def route_predictions(X, labels):
        preds = np.zeros(len(X))
        for i, label in enumerate(labels):
            if label == 0:
                preds[i] = reg_low.predict(X[i].reshape(1, -1))[0]
            else:
                preds[i] = reg_high.predict(X[i].reshape(1, -1))[0]
        return preds

    # 1️⃣ Ideal routing (true labels)
    y_train_pred_true = route_predictions(X_train, y_class_train)
    y_test_pred_true  = route_predictions(X_test,  y_class_test)

    # 2️⃣ Classifier routing (predicted labels)
    y_class_pred_train = clf.predict(X_train)
    y_class_pred_test  = clf.predict(X_test)
    y_train_pred_clf = route_predictions(X_train, y_class_pred_train)
    y_test_pred_clf  = route_predictions(X_test,  y_class_pred_test)

    # === Evaluate ===
    print("\n--- Metrics ---")
    print("Train MAPE (ideal routing):      ", mean_absolute_percentage_error(y_train, y_train_pred_true))
    print("Test  MAPE (ideal routing):      ", mean_absolute_percentage_error(y_test,  y_test_pred_true))
    print("Train MAPE (classifier routing): ", mean_absolute_percentage_error(y_train, y_train_pred_clf))
    print("Test  MAPE (classifier routing): ", mean_absolute_percentage_error(y_test,  y_test_pred_clf))

    # === Confusion Matrix ===
    cm = confusion_matrix(y_class_test, y_class_pred_test)
    ConfusionMatrixDisplay(cm, display_labels=["≤ threshold", "> threshold"]).plot(cmap="Blues")
    plt.title("Classifier Confusion Matrix (Test)")
    plt.show()

    # === Scatter Plots ===
    def plot_scatter(y_true, y_pred, y_class, split, mode):
        plt.figure(figsize=(6,6))
        for label, color in zip([0,1], ["blue","red"]):
            mask = (y_class == label)
            plt.scatter(y_true[mask], y_pred[mask], alpha=0.6, c=color, label=f"Class {label}")
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        plt.plot([min_val, max_val],[min_val,max_val],'k--')
        plt.xlabel("Actual"); plt.ylabel("Predicted")
        plt.title(f"{split} Set – {mode}")
        plt.legend(); plt.gca().set_aspect("equal","box")
        plt.show()

    # Train/Test for both routing types
    plot_scatter(y_train, y_train_pred_true, y_class_train, "Train", "Ideal Routing")
    plot_scatter(y_test,  y_test_pred_true,  y_class_test,  "Test",  "Ideal Routing")
    plot_scatter(y_train, y_train_pred_clf,  y_class_train, "Train", "Classifier Routing")
    plot_scatter(y_test,  y_test_pred_clf,   y_class_test,  "Test",  "Classifier Routing")

    return clf, reg_low, reg_high, y_train_pred_true, y_test_pred_true, y_train_pred_clf, y_test_pred_clf


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Training and test sets (already normalized!)
        train_df = data['train']
        test_df  = data['test']

        # Feature columns (exclude target)
        num_cols_norm = [c for c in train_df.columns if c != target_col]

        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        clf, reg_low, reg_higher, y_train_pred, y_test_pred, y_train_pred_clf, y_test_pred_clf = classify_and_regress_compare(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            threshold=-0.5,
            #clf_C=0.3,                # smaller = stronger regularization
            #clf_class_weight="balanced",
            #clf_threshold=0.4         # adjust decision threshold
        )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, confusion_matrix, ConfusionMatrixDisplay

def classify_and_regress(X_train, y_train, X_test, y_test, threshold=2):
    """
    Step 1: Train classifier (y <= threshold vs y > threshold).
    Step 2: Train two separate regressors (one per class).
    Step 3: Scatter plots for train and test.
    Step 4: Show confusion matrix on test set + sanity check.
    """
    # Ensure arrays
    y_train = np.asarray(y_train)
    y_test  = np.asarray(y_test)

    # === Define classes based on actual values ===
    y_class_train = (y_train > threshold).astype(int)
    y_class_test  = (y_test  > threshold).astype(int)

    # === Train classifier ===
    clf = LogisticRegression(class_weight="balanced", max_iter=1000)
    clf.fit(X_train, y_class_train)

    # === Train regressors ===
    reg_low  = LinearRegression()  # for y <= threshold
    reg_high = LinearRegression()  # for y > threshold

    reg_low.fit(X_train[y_class_train == 0], y_train[y_class_train == 0])
    reg_high.fit(X_train[y_class_train == 1], y_train[y_class_train == 1])

    # === Predict class labels ===
    y_class_pred_train = clf.predict(X_train)
    y_class_pred_test  = clf.predict(X_test)

    # === Route predictions ===
    def route_predictions(X, y_class_pred):
        preds = np.zeros(len(X))
        for i, label in enumerate(y_class_pred):
            if label == 0:
                preds[i] = reg_low.predict(X[i].reshape(1, -1))[0]
            else:
                preds[i] = reg_high.predict(X[i].reshape(1, -1))[0]
        return preds

    y_train_pred = route_predictions(X_train, y_class_pred_train)
    y_test_pred  = route_predictions(X_test,  y_class_pred_test)

    # === Evaluate ===
    train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
    test_mape  = mean_absolute_percentage_error(y_test,  y_test_pred)
    print(f"Train MAPE: {train_mape:.4f}")
    print(f"Test  MAPE: {test_mape:.4f}")

    # === Confusion Matrix (Test) ===
    cm = confusion_matrix(y_class_test, y_class_pred_test)
    print("\nConfusion Matrix raw values:")
    print(cm)
    print(f" Sum of confusion matrix cells = {cm.sum()}, "
          f"Number of test samples = {len(y_class_test)}")

    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=["<= threshold", "> threshold"])
    disp.plot(cmap="Blues")
    plt.title("Classifier Confusion Matrix (Test)")
    plt.show()

    # === Scatter plots ===
    def plot_scatter(y_true, y_pred, y_class_true, split="Train"):
        plt.figure(figsize=(6,6))
        for label, color in zip([0,1], ["blue","red"]):
            mask = (y_class_true == label)
            plt.scatter(y_true[mask], y_pred[mask], alpha=0.5, s=20,
                        c=color, label=f"Class {label}")
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        plt.plot([min_val, max_val],[min_val,max_val],'k--')
        plt.xlabel("Actual"); plt.ylabel("Predicted")
        plt.title(f"Actual vs Predicted ({split} Set)")
        plt.legend()
        plt.gca().set_aspect("equal","box")
        plt.show()

    plot_scatter(y_train, y_train_pred, y_class_train, split="Train")
    plot_scatter(y_test,  y_test_pred,  y_class_test,  split="Test")

    return clf, reg_low, reg_high, y_train_pred, y_test_pred


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Training and test sets (already normalized!)
        train_df = data['train']
        test_df  = data['test']

        # Feature columns (exclude target)
        num_cols_norm = [c for c in train_df.columns if c != target_col]

        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        clf, reg_near, reg_away, y_train_pred, y_test_pred = classify_and_regress(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            threshold=-0.5,
            #clf_C=0.3,                # smaller = stronger regularization
            #clf_class_weight="balanced",
            #clf_threshold=0.4         # adjust decision threshold
        )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, confusion_matrix, ConfusionMatrixDisplay

def classify_and_regress_with_higher_prob(X_train, y_train, X_test, y_test, threshold=-2):
    """
    Step 1: Train classifier (y <= threshold vs y > threshold).
    Step 2: Train two separate regressors (one per class).
    Step 3: Scatter plots for train and test.
    Step 4: Show confusion matrix on test set + sanity check.
    """
    # Ensure arrays
    y_train = np.asarray(y_train)
    y_test  = np.asarray(y_test)

    # === Define classes based on actual values ===
    y_class_train = (y_train > threshold).astype(int)

    y_class_test  = (y_test  > threshold).astype(int)

    # === Train classifier ===
    clf = LogisticRegression(class_weight="balanced", max_iter=1000)
    clf.fit(X_train, y_class_train)

    # === Train regressors ===
    reg_low  = LinearRegression()  # for y <= threshold
    reg_high = LinearRegression()  # for y > threshold

    reg_low.fit(X_train[y_class_train == 0], y_train[y_class_train == 0])
    reg_high.fit(X_train[y_class_train == 1], y_train[y_class_train == 1])

    # === Predict class labels ===
    y_class_pred_train = clf.predict(X_train)
    #y_prob_train = clf.predict_proba(X_train)[:, 1]   # probability of > threshold
    #y_class_pred_train = (y_prob_train > 0.85).astype(int)

    #y_class_pred_test  = clf.predict(X_test)

    y_prob = clf.predict_proba(X_test)[:, 1]   # probability of > threshold
    y_class_pred_test = (y_prob > 0.75).astype(int)


    # === Route predictions ===
    def route_predictions(X, y_class_pred):
        preds = np.zeros(len(X))
        for i, label in enumerate(y_class_pred):
            if label == 0:
                preds[i] = reg_low.predict(X[i].reshape(1, -1))[0]
            else:
                preds[i] = reg_high.predict(X[i].reshape(1, -1))[0]
        return preds

    y_train_pred = route_predictions(X_train, y_class_pred_train)
    y_test_pred  = route_predictions(X_test,  y_class_pred_test)

    # === Evaluate ===
    train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
    test_mape  = mean_absolute_percentage_error(y_test,  y_test_pred)
    print(f"Train MAPE: {train_mape:.4f}")
    print(f"Test  MAPE: {test_mape:.4f}")

    # === Confusion Matrix (Test) ===
    cm = confusion_matrix(y_class_test, y_class_pred_test)
    print("\nConfusion Matrix raw values:")
    print(cm)
    print(f" Sum of confusion matrix cells = {cm.sum()}, "
          f"Number of test samples = {len(y_class_test)}")

    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=["<= threshold", "> threshold"])
    disp.plot(cmap="Blues")
    plt.title("Classifier Confusion Matrix (Test)")
    plt.show()

    # === Scatter plots ===
    def plot_scatter(y_true, y_pred, y_class_true, split="Train"):
        plt.figure(figsize=(6,6))
        for label, color in zip([0,1], ["blue","red"]):
            mask = (y_class_true == label)
            plt.scatter(y_true[mask], y_pred[mask], alpha=0.5, s=20,
                        c=color, label=f"Class {label}")
        min_val = min(y_true.min(), y_pred.min())
        max_val = max(y_true.max(), y_pred.max())
        plt.plot([min_val, max_val],[min_val,max_val],'k--')
        plt.xlabel("Actual"); plt.ylabel("Predicted")
        plt.title(f"Actual vs Predicted ({split} Set)")
        plt.legend()
        plt.gca().set_aspect("equal","box")
        plt.show()

    plot_scatter(y_train, y_train_pred, y_class_train, split="Train")
    plot_scatter(y_test,  y_test_pred,  y_class_test,  split="Test")

    return clf, reg_low, reg_high, y_train_pred, y_test_pred

In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Training and test sets (already normalized!)
        train_df = data['train']
        test_df  = data['test']

        # Feature columns (exclude target)
        num_cols_norm = [c for c in train_df.columns if c != target_col]

        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        clf, reg_near, reg_away, y_train_pred, y_test_pred = classify_and_regress_with_higher_prob(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            threshold=-0.3,
            #clf_C=0.3,                # smaller = stronger regularization
            #clf_class_weight="balanced",
            #clf_threshold=0.4         # adjust decision threshold
        )

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def try_non_linear_classifier(X_train, y_train, X_test, y_test, threshold=0.0):
    """
    Compare Logistic Regression vs Random Forest classifier performance
    and show confusion matrices.
    """
    # Convert regression target into binary labels
    y_train_class = (y_train > threshold).astype(int)
    y_test_class  = (y_test  > threshold).astype(int)

    results = {}

    # --- Logistic Regression ---
    log_clf = LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
    log_clf.fit(X_train, y_train_class)
    log_acc = log_clf.score(X_test, y_test_class)
    log_pred = log_clf.predict(X_test)

    cm_log = confusion_matrix(y_test_class, log_pred)
    disp_log = ConfusionMatrixDisplay(cm_log, display_labels=["<= threshold", "> threshold"])
    disp_log.plot(cmap="Blues")
    plt.title("Logistic Regression Confusion Matrix")
    plt.show()

    results["Logistic Regression"] = {
        "accuracy": log_acc,
        "confusion_matrix": cm_log
    }

    # --- Random Forest ---
    rf_clf = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        class_weight="balanced_subsample",
        random_state=42
    )
    rf_clf.fit(X_train, y_train_class)
    rf_acc = rf_clf.score(X_test, y_test_class)
    rf_pred = rf_clf.predict(X_test)

    cm_rf = confusion_matrix(y_test_class, rf_pred)
    disp_rf = ConfusionMatrixDisplay(cm_rf, display_labels=["<= threshold", "> threshold"])
    disp_rf.plot(cmap="Greens")
    plt.title("Random Forest Confusion Matrix")
    plt.show()

    results["Random Forest"] = {
        "accuracy": rf_acc,
        "confusion_matrix": cm_rf
    }

    # Print results summary
    print("\n=== Results Summary ===")
    for model, metrics in results.items():
        print(f"{model}:")
        print(f"  Accuracy: {metrics['accuracy']:.3f}")
        print(f"  Confusion Matrix:\n{metrics['confusion_matrix']}\n")

    return log_clf, rf_clf


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Training and test sets (already normalized!)
        train_df = data['train']
        test_df  = data['test']

        # Feature columns (exclude target)
        num_cols_norm = [c for c in train_df.columns if c != target_col]

        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        log_clf, rlf_clf = try_non_linear_classifier(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            threshold=-0.3,
            #clf_C=0.3,                # smaller = stronger regularization
            #clf_class_weight="balanced",
            #clf_threshold=0.4         # adjust decision threshold
        )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    fbeta_score, make_scorer, confusion_matrix, precision_recall_curve, roc_curve, auc
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# --------------------------
# labeling (positive = under-threat)
# --------------------------
def make_binary_labels(y, threshold):
    # 1 = under-threat (<= threshold), 0 = safe (> threshold)
    return (np.asarray(y) <= threshold).astype(int)

# --------------------------
# sweep probability threshold to pick the best cut
# --------------------------
def find_best_threshold(y_true_bin, y_prob, beta=2.0, w_fn=3.0, w_fp=1.0):
    """
    Search thresholds in [0,1] to (a) maximize F_beta (recall-weighted)
    and (b) minimize cost = w_fn*FN + w_fp*FP. Returns both choices.
    """
    y_true_bin = np.asarray(y_true_bin).astype(int)
    thresholds = np.linspace(0.01, 0.99, 99)

    best_fbeta, best_fbeta_thr = -1.0, 0.5
    best_cost, best_cost_thr   = 1e18, 0.5

    fbeta_curve, cost_curve = [], []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        # F_beta (positive = under-threat = 1)
        fbeta = fbeta_score(y_true_bin, y_pred, beta=beta, pos_label=1)
        fbeta_curve.append(fbeta)

        # cost
        tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred, labels=[0,1]).ravel()
        cost = w_fn*fn + w_fp*fp
        cost_curve.append(cost)

        if fbeta > best_fbeta:
            best_fbeta, best_fbeta_thr = fbeta, t
        if cost < best_cost:
            best_cost, best_cost_thr = cost, t

    # Plot PR curve (nice to see too)
    prec, rec, _ = precision_recall_curve(y_true_bin, y_prob, pos_label=1)
    plt.figure(figsize=(5,5))
    plt.plot(rec, prec)
    plt.xlabel("Recall (Under-threat)")
    plt.ylabel("Precision (Under-threat)")
    plt.title("Precision–Recall curve (test probs)")
    plt.show()

    # Plot F_beta & cost over thresholds
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(thresholds, fbeta_curve)
    plt.axvline(best_fbeta_thr, ls='--', c='k', label=f"best Fβ@{best_fbeta_thr:.2f}")
    plt.xlabel("Threshold"); plt.ylabel(f"Fβ (β={beta})"); plt.legend()

    plt.subplot(1,2,2)
    plt.plot(thresholds, cost_curve)
    plt.axvline(best_cost_thr, ls='--', c='k', label=f"min cost@{best_cost_thr:.2f}")
    plt.xlabel("Threshold"); plt.ylabel(f"Cost = {w_fn}·FN + {w_fp}·FP"); plt.legend()
    plt.tight_layout()
    plt.show()

    return {"best_fbeta_thr": best_fbeta_thr, "best_cost_thr": best_cost_thr,
            "best_fbeta": best_fbeta, "best_cost": best_cost}


In [ ]:
def grid_search_classifier(X_train, y_train_continuous, threshold,
                           model_name="logreg", param_grid=None,
                           cv_splits=5, beta=2.0, class_weight="balanced"):
    """
    Turns y into binary (positive = under-threat), runs GridSearchCV,
    returns best classifier and fitted GridSearchCV object.
    """
    y_bin = make_binary_labels(y_train_continuous, threshold)

    # default grids
    if model_name == "logreg":
        clf = LogisticRegression(max_iter=2000, class_weight=class_weight)
        param_grid = param_grid or {
            "C": [0.05, 0.1, 0.3, 1, 3, 10],
            "penalty": ["l2"],
            "solver": ["lbfgs"]
        }
    elif model_name == "rf":
        clf = RandomForestClassifier(class_weight=class_weight, n_jobs=-1, random_state=42)
        param_grid = param_grid or {
            "n_estimators": [200, 400],
            "max_depth": [None, 10, 20],
            "min_samples_leaf": [1, 2, 5]
        }
    elif model_name == "gb":
        clf = GradientBoostingClassifier(random_state=42)
        param_grid = param_grid or {
            "n_estimators": [200, 400],
            "learning_rate": [0.05, 0.1],
            "max_depth": [1, 2, 3]
        }
    elif model_name == "svc":
        clf = SVC(probability=True, class_weight=class_weight, random_state=42)
        param_grid = param_grid or {
            "C": [0.3, 1, 3, 10],
            "kernel": ["rbf", "linear"],
            "gamma": ["scale", "auto"]
        }
    else:
        raise ValueError("Unknown model_name")

    scorer = make_scorer(fbeta_score, beta=beta, pos_label=1)
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
    gs = GridSearchCV(clf, param_grid=param_grid, scoring=scorer, cv=cv, n_jobs=-1, refit=True)
    gs.fit(X_train, y_bin)

    print(f"[{model_name}] best params:", gs.best_params_)
    print(f"[{model_name}] best CV Fβ (β={beta}): {gs.best_score_:.3f}")

    return gs.best_estimator_, gs


In [ ]:
from sklearn.metrics import mean_absolute_percentage_error, classification_report, ConfusionMatrixDisplay

def fit_two_regressors(X_train, y_train, y_train_bin, reg_kind="linear"):
    # choose regression model (you can switch to Ridge/Lasso if you prefer)
    if reg_kind == "linear":
        reg0 = LinearRegression()
        reg1 = LinearRegression()
    elif reg_kind == "ridge":
        reg0 = Ridge(alpha=1.0)
        reg1 = Ridge(alpha=1.0)
    else:
        raise ValueError("reg_kind must be 'linear' or 'ridge'")

    reg0.fit(X_train[y_train_bin == 0], y_train[y_train_bin == 0])  # safe
    reg1.fit(X_train[y_train_bin == 1], y_train[y_train_bin == 1])  # under-threat
    return reg0, reg1

def routed_predictions(clf, thr, regs, X):
    reg0, reg1 = regs
    prob = clf.predict_proba(X)[:, 1]           # P(under-threat)
    cls  = (prob >= thr).astype(int)
    yhat = np.empty(len(prob), dtype=float)
    mask1 = (cls==1); mask0 = ~mask1
    if mask0.any(): yhat[mask0] = reg0.predict(X[mask0])
    if mask1.any(): yhat[mask1] = reg1.predict(X[mask1])
    return yhat, cls, prob


In [ ]:
from sklearn.metrics import roc_auc_score

def evaluate_pipeline(y_train, y_train_pred, y_test, y_test_pred,
                      y_train_bin, y_pred_train_bin, y_test_bin, y_pred_test_bin,
                      y_prob_test, model_label, threshold):
    # Regression
    tr_mape = mean_absolute_percentage_error(y_train, y_train_pred)
    te_mape = mean_absolute_percentage_error(y_test,  y_test_pred)
    print(f"\n[{model_label}] Train MAPE: {tr_mape:.4f} | Test MAPE: {te_mape:.4f}")

    # Confusion matrix + report (test)
    cm = confusion_matrix(y_test_bin, y_pred_test_bin, labels=[0,1])
    print("\nConfusion matrix (rows=true, cols=pred):\n", cm)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["safe", "under-threat"])
    disp.plot(cmap="Blues")
    plt.title(f"{model_label} – Confusion Matrix (thr={threshold:.2f})")
    plt.show()

    print("\nClassification report (test):")
    print(classification_report(y_test_bin, y_pred_test_bin, target_names=["safe", "under-threat"], digits=3))

    # ROC-AUC + curves
    if y_prob_test is not None:
        fpr, tpr, _ = roc_curve(y_test_bin, y_prob_test, pos_label=1)
        roc_auc = auc(fpr, tpr)
        print(f"ROC AUC (test): {roc_auc:.3f}")
        plt.figure(figsize=(5,5))
        plt.plot(fpr, tpr, label=f"AUC={roc_auc:.2f}")
        plt.plot([0,1],[0,1],'k--'); plt.legend(); plt.xlabel("FPR"); plt.ylabel("TPR")
        plt.title(f"{model_label} – ROC Curve")
        plt.show()

        prec, rec, _ = precision_recall_curve(y_test_bin, y_prob_test, pos_label=1)
        plt.figure(figsize=(5,5))
        plt.plot(rec, prec)
        plt.xlabel("Recall (under-threat)"); plt.ylabel("Precision (under-threat)")
        plt.title(f"{model_label} – Precision–Recall Curve")
        plt.show()

    # Scatter plots
    def scatter(y_true, y_pred, y_bin, split):
        plt.figure(figsize=(6,6))
        for label, color, name in [(0,"tab:blue","safe"), (1,"tab:red","under-threat")]:
            m = (y_bin==label)
            if m.any(): plt.scatter(y_true[m], y_pred[m], s=16, alpha=0.5, c=color, label=name)
        lo, hi = float(min(y_true.min(), y_pred.min())), float(max(y_true.max(), y_pred.max()))
        plt.plot([lo,hi],[lo,hi],'k--'); plt.gca().set_aspect("equal","box")
        plt.xlabel("Actual"); plt.ylabel("Predicted")
        plt.title(f"{model_label} – {split}")
        plt.legend(); plt.show()

    scatter(y_train, y_train_pred, y_train_bin, "Train")
    scatter(y_test,  y_test_pred,  y_test_bin,  "Test")


In [ ]:
def run_router_once(X_train, y_train, X_test, y_test,
                    threshold,
                    model_name="logreg", param_grid=None,
                    beta=2.0, w_fn=3.0, w_fp=1.0,
                    reg_kind="linear", label=""):
    # labels
    y_train_bin = make_binary_labels(y_train, threshold)
    y_test_bin  = make_binary_labels(y_test,  threshold)

    # classifier search
    clf, gs = grid_search_classifier(
        X_train, y_train, threshold,
        model_name=model_name, param_grid=param_grid,
        beta=beta, class_weight="balanced"
    )

    # train regressors per class
    reg0, reg1 = fit_two_regressors(X_train, y_train, y_train_bin, reg_kind=reg_kind)

    # decide threshold on test probs (or you can use CV/val split!)
    y_prob_test = clf.predict_proba(X_test)[:,1]
    thr_info = find_best_threshold(y_test_bin, y_prob_test, beta=beta, w_fn=w_fn, w_fp=w_fp)
    thr_star = thr_info["best_cost_thr"]  # or "best_fbeta_thr"

    # predictions (train/test) routed by chosen thr
    y_train_pred, y_pred_train_bin, _ = routed_predictions(clf, thr_star, (reg0,reg1), X_train)
    y_test_pred,  y_pred_test_bin,  y_prob_test = routed_predictions(clf, thr_star, (reg0,reg1), X_test)

    evaluate_pipeline(
        y_train, y_train_pred,
        y_test,  y_test_pred,
        y_train_bin, y_pred_train_bin,
        y_test_bin,  y_pred_test_bin,
        y_prob_test,
        model_label=f"{label or model_name} (thr={thr_star:.2f})", threshold=threshold
    )

    return {"clf": clf, "reg0": reg0, "reg1": reg1, "thr": thr_star, "gs": gs}


In [ ]:
# Example loop
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")
        train_df = data["train"]; test_df = data["test"]

        # choose columns (already normalized in splits_norm)
        num_cols_norm = [c for c in train_df.columns if c != target_col]
        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        # your groundwater threshold
        thr = -0.2  # <-- set this

        # 1) Logistic (linear) router
        run_router_once(
            X_train, y_train, X_test, y_test,
            threshold=thr,
            model_name="logreg",
            param_grid={"C":[0.05,0.1,0.3,1,3,10], "solver":["lbfgs"], "penalty":["l2"]},
            beta=2.0, w_fn=3.0, w_fp=1.0,    # emphasize catching under-threat
            reg_kind="ridge",
            label=f"{aquifer}-{season_label}: Logistic"
        )

        # 2) Random Forest router (non-linear)
        run_router_once(
            X_train, y_train, X_test, y_test,
            threshold=thr,
            model_name="rf",
            param_grid={"n_estimators":[300,500],
                        "max_depth":[None,10,20],
                        "min_samples_leaf":[1,2,5]},
            beta=2.0, w_fn=3.0, w_fp=1.0,
            reg_kind="ridge",
            label=f"{aquifer}-{season_label}: RF"
        )

        # 3) Gradient Boosting router (non-linear)
        run_router_once(
            X_train, y_train, X_test, y_test,
            threshold=thr,
            model_name="gb",
            param_grid={"n_estimators":[300,500],
                        "learning_rate":[0.05,0.1],
                        "max_depth":[2,3]},
            beta=2.0, w_fn=3.0, w_fp=1.0,
            reg_kind="ridge",
            label=f"{aquifer}-{season_label}: GB"
        )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.metrics import confusion_matrix
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Tversky definition
def tversky_from_counts(tp, fp, fn, alpha=0.5, beta=0.5, eps=1e-9):
    return tp / (tp + alpha * fp + beta * fn + eps)

from sklearn.base import BaseEstimator, ClassifierMixin

class ThresholdWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, est=None, threshold=0.5):
        self.est = est
        self.threshold = float(threshold)

    def fit(self, X, y):
        self.est.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.est.predict_proba(X)

    def predict(self, X):
        p = self.predict_proba(X)[:, 1]
        return (p > self.threshold).astype(int)

    # expose params in a namespaced way so sklearn.clone works
    def get_params(self, deep=True):
        params = {"est": self.est, "threshold": self.threshold}
        if deep and hasattr(self.est, "get_params"):
            for k, v in self.est.get_params(deep=True).items():
                params[f"est__{k}"] = v
        return params

    def set_params(self, **params):
        for k, v in params.items():
            if k == "threshold":
                self.threshold = float(v)
            elif k == "est":
                self.est = v
            elif k.startswith("est__"):
                sub_k = k.split("est__", 1)[1]
                self.est.set_params(**{sub_k: v})
            else:
                # ignore unknown keys
                pass
        return self


def oof_confusion_matrix(estimator, X, y, params, cv_splits=5, random_state=42):
    """
    Out of fold predictions only on train to build a confusion matrix
    Honest evaluation for a given parameter set
    """
    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    y_pred = np.zeros_like(y, dtype=int)

    for tr, va in skf.split(X, y):
        est = clone(estimator)
        est.set_params(**params)
        est.fit(X[tr], y[tr])
        y_pred[va] = est.predict(X[va])

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    return cm

def print_cm(cm, title=None):
    tn, fp, fn, tp = cm.ravel()
    if title:
        print(title)
    print("    Confusion matrix [rows true, cols pred]")
    print(f"        [[TN={tn:4d}, FP={fp:4d}]")
    print(f"         [FN={fn:4d}, TP={tp:4d}]]")
    return tp, fp, fn, tn

def plot_cm(cm, title):
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar(fraction=0.046, pad=0.04)
    ticks = np.arange(2)
    plt.xticks(ticks, ["<= thr", "> thr"])
    plt.yticks(ticks, ["<= thr", "> thr"])
    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i, j], ha="center", va="center")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

def classifier_tversky_grid_and_test(
    X_train, y_train_cont,
    X_test, y_test_cont,
    threshold_value=-2.0,
    alpha=0.5, beta=0.5,
    cv_splits=5, random_state=42,
    plot_each_matrix=False
):
    """
    Runs grid search with Tversky objective
    Prints the confusion matrix for every hyperparameter on train via out of fold predictions
    Picks the best Tversky, refits on full train, and prints the confusion matrix on test
    """

    # Build binary labels from continuous target
    y_train_bin = (np.asarray(y_train_cont) > threshold_value).astype(int)
    y_test_bin  = (np.asarray(y_test_cont)  > threshold_value).astype(int)

    # Default models and grids
    models_and_grids = [
      (
          "LogisticRegression",
          ThresholdWrapper(
              est=LogisticRegression(max_iter=2000, solver="lbfgs", random_state=random_state),
              threshold=0.5
          ),
          {
              "est__C": [0.05, 0.1, 0.5, 1.0, 5.0],
              "threshold": [0.3, 0.4, 0.5, 0.6, 0.7]
          }
      ),
      (
          "RandomForest",
          ThresholdWrapper(
              est=RandomForestClassifier(n_estimators=400, random_state=random_state, n_jobs=-1),
              threshold=0.5
          ),
          {
              "est__max_depth": [None, 6, 12],
              "est__min_samples_leaf": [1, 3, 5],
              "est__max_features": ["sqrt", "log2"],
              "threshold": [0.3, 0.4, 0.5, 0.6, 0.7]
          }
      ),
      (
          "GradientBoosting",
          ThresholdWrapper(
              est=GradientBoostingClassifier(random_state=random_state),
              threshold=0.5
          ),
          {
              "est__n_estimators": [150, 300],
              "est__learning_rate": [0.05, 0.1],
              "est__max_depth": [2, 3],
              "threshold": [0.3, 0.4, 0.5, 0.6, 0.7]
          }
      ),
  ]


    all_rows = []
    print(f"\nTversky objective with alpha={alpha} and beta={beta}")
    for name, base_est, grid in models_and_grids:
        print(f"\nModel family: {name}")
        for params in ParameterGrid(grid):
            cm = oof_confusion_matrix(base_est, X_train, y_train_bin, params,
                                      cv_splits=cv_splits, random_state=random_state)
            tn, fp, fn, tp = cm.ravel()
            tv = tversky_from_counts(tp, fp, fn, alpha=alpha, beta=beta)
            all_rows.append({
                "model": name,
                "params": params,
                "cm": cm,
                "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
                "tversky": float(tv),
            })
            tp_, fp_, fn_, tn_ = print_cm(cm, title=f"  params={params}  Tversky={tv:.4f}")
            if plot_each_matrix:
                plt.figure(figsize=(3.8, 3.8))
                plot_cm(cm, title=f"{name}\n{params}\nTversky={tv:.3f}")
                plt.tight_layout()
                plt.show()

    # Select best by Tversky
    all_rows.sort(key=lambda r: r["tversky"], reverse=True)
    best = all_rows[0]
    print("\nBest by Tversky on out of fold train")
    print(f"  Model:  {best['model']}")
    print(f"  Params: {best['params']}")
    print(f"  Score:  {best['tversky']:.4f}")
    print_cm(best["cm"])

    # Refit best on full train and evaluate once on test
    print("\nRefit best on full train and evaluate on test")
    # Find the matching estimator blueprint
    name_to_est = {m[0]: m[1] for m in models_and_grids}
    final_est = clone(name_to_est[best["model"]])
    final_est.set_params(**best["params"])
    final_est.fit(X_train, y_train_bin)
    y_pred_test = final_est.predict(X_test)
    cm_test = confusion_matrix(y_test_bin, y_pred_test, labels=[0, 1])
    tp_t, fp_t, fn_t, tn_t = print_cm(cm_test, title="  Test confusion matrix")
    tv_test = tversky_from_counts(tp_t, fp_t, fn_t, alpha=alpha, beta=beta)
    print(f"  Test Tversky={tv_test:.4f}")

    # Plot test matrix
    plt.figure(figsize=(3.8, 3.8))
    plot_cm(cm_test, title=f"Test — {best['model']}\n{best['params']}\nTversky={tv_test:.3f}")
    plt.tight_layout()
    plt.show()

    return {
        "all_rows_sorted": all_rows,   # every hyperparameter with its OOF confusion matrix and Tversky
        "best": best,                  # best on OOF
        "cm_test": cm_test,            # test confusion matrix of the refit best model
        "tversky_test": tv_test
    }


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Split into train/test
        train_df = data["train"]
        test_df  = data["test"]

        # Feature matrix and target vector
        num_cols_norm = [c for c in train_df.columns if c != target_col]
        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        # --- call the classifier-only function ---
        result = classifier_tversky_grid_and_test(
            X_train=X_train,
            y_train_cont=y_train,
            X_test=X_test,
            y_test_cont=y_test,
            threshold_value=-0.2,  # same as before
            alpha=0.7,             # weight for false positives
            beta=0.3,              # weight for false negatives
            cv_splits=5,
            plot_each_matrix=True
        )

        # result contains everything if you want to store or log it
        best_model = result["best"]["model"]
        best_params = result["best"]["params"]
        print(f"\nBest classifier for {aquifer} – {season_label}: "
              f"{best_model} with {best_params} "
              f"(Test Tversky={result['tversky_test']:.4f})")


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        # Split into train/test
        train_df = data["train"]
        test_df  = data["test"]

        # Feature matrix and target vector
        num_cols_norm = [c for c in train_df.columns if c != target_col]
        X_train = train_df[num_cols_norm].values
        y_train = train_df[target_col].values
        X_test  = test_df[num_cols_norm].values
        y_test  = test_df[target_col].values

        # --- call the classifier-only function ---
        result = classifier_tversky_grid_and_test(
            X_train=X_train,
            y_train_cont=y_train,
            X_test=X_test,
            y_test_cont=y_test,
            threshold_value=-2.0,  # same as before
            alpha=0.7,             # weight for false positives
            beta=0.3,              # weight for false negatives
            cv_splits=5,
            plot_each_matrix=True
        )

        # result contains everything if you want to store or log it
        best_model = result["best"]["model"]
        best_params = result["best"]["params"]
        print(f"\nBest classifier for {aquifer} – {season_label}: "
              f"{best_model} with {best_params} "
              f"(Test Tversky={result['tversky_test']:.4f})")


In [ ]:
import itertools
import numpy as np
import pandas as pd

from statsmodels.api import OLS, add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

def compute_vif_df(X: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a DataFrame with VIF per column in X.
    Adds a constant inside and drops it from the returned table.
    """
    Xc = add_constant(X, has_constant="add")
    vif_vals = []
    cols = Xc.columns.tolist()
    for i, col in enumerate(cols):
        vif_vals.append(variance_inflation_factor(Xc.values, i))
    vif_df = pd.DataFrame({"feature": cols, "vif": vif_vals})
    vif_df = vif_df[vif_df["feature"] != "const"].reset_index(drop=True)
    return vif_df

def fit_adj_r2(X: pd.DataFrame, y: np.ndarray) -> float:
    """
    Returns adjusted R2 from statsmodels OLS on X predicting y.
    """
    Xc = add_constant(X, has_constant="add")
    model = OLS(y, Xc, hasconst=True).fit()
    return float(model.rsquared_adj)

def search_low_vif_subsets(
    df: pd.DataFrame,
    target_col: str,
    candidate_cols: list,
    threshold_value: float,
    vif_limit: float = 5.0,
    min_features: int = 4,
    verbose: bool = True
):
    """
    Exhaustive search over all combinations from full set down to min_features,
    on the subset where y <= threshold_value.

    Keeps only combos where every individual VIF < vif_limit.
    Ranks by min max VIF, then min mean VIF, then max adjusted R2.

    Returns a dict with:
      best_result: dict with keys combo, vif_df, max_vif, mean_vif, adj_r2, k, n_kept
      all_valid: list of dicts for all valid combos considered
    """
    # subset rows where y <= threshold
    sub = df[df[target_col] <= threshold_value].copy()
    y = sub[target_col].values

    p = len(candidate_cols)
    if p < min_features:
        raise ValueError(f"Need at least {min_features} candidate features, got {p}")

    all_valid = []
    if verbose:
        print(f"Total candidates: {p}. Searching sizes from {p} down to {min_features} on {len(sub)} rows")

    for k in range(p, min_features - 1, -1):
        if verbose:
            print(f"\nEvaluating combinations of size k = {k}")
        for combo in itertools.combinations(candidate_cols, k):
            X = sub[list(combo)]
            # skip degenerate columns
            if X.shape[1] != k:
                continue
            # compute VIFs
            vif_df = compute_vif_df(X)
            max_vif = float(vif_df["vif"].max())
            if max_vif >= vif_limit:
                continue  # discard this combo

            mean_vif = float(vif_df["vif"].mean())
            adj_r2 = fit_adj_r2(X, y)

            all_valid.append({
                "k": k,
                "combo": combo,
                "vif_df": vif_df,
                "max_vif": max_vif,
                "mean_vif": mean_vif,
                "adj_r2": adj_r2
            })

        if verbose:
            kept = sum(1 for r in all_valid if r["k"] == k)
            print(f"  kept {kept} combos at size {k} with all VIF < {vif_limit}")

    if not all_valid:
        if verbose:
            print("No combination satisfied the VIF constraint")
        return {"best_result": None, "all_valid": []}

    # rank
    all_valid.sort(key=lambda r: (r["max_vif"], r["mean_vif"], -r["adj_r2"]))
    best = all_valid[0]
    n_kept = sum(1 for r in all_valid if r["k"] == best["k"])
    best["n_kept"] = n_kept

    if verbose:
        print("\nBest subset by VIF ranking")
        print(f"  size k = {best['k']}")
        print(f"  combo = {best['combo']}")
        print(f"  max VIF = {best['max_vif']:.3f}, mean VIF = {best['mean_vif']:.3f}, adj R2 = {best['adj_r2']:.4f}")
        print("  VIF table:")
        print(best["vif_df"].to_string(index=False))

    return {"best_result": best, "all_valid": all_valid}

In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        train_df = data["train"]
        num_cols_norm = [c for c in train_df.columns if c != target_col]

        res = search_low_vif_subsets(
            df=train_df,
            target_col=target_col,
            candidate_cols=num_cols_norm,
            threshold_value=-0.2,  # your current split
            vif_limit=5.0,
            min_features=4,
            verbose=True
        )

        best = res["best_result"]
        if best is None:
            print("No valid subset with all VIF below 5")
            continue

        # If you want the chosen features for regression later
        chosen_features = list(best["combo"])
        print(f"Chosen features for low side regression: {chosen_features}")


In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.api import OLS, add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --- helpers ---
def compute_vif_df(X: pd.DataFrame) -> pd.DataFrame:
    Xc = add_constant(X, has_constant="add")
    vif_vals = []
    cols = Xc.columns.tolist()
    for i, col in enumerate(cols):
        vif_vals.append(variance_inflation_factor(Xc.values, i))
    vif_df = pd.DataFrame({"feature": cols, "vif": vif_vals})
    vif_df = vif_df[vif_df["feature"] != "const"].reset_index(drop=True)
    return vif_df

def fit_adj_r2(X: pd.DataFrame, y: np.ndarray) -> float:
    Xc = add_constant(X, has_constant="add")
    model = OLS(y, Xc, hasconst=True).fit()
    return float(model.rsquared_adj)

def search_low_vif_and_plot_regressions(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    target_col: str,
    candidate_cols: list,
    threshold_value: float,
    vif_limit: float = 5.0,
    min_features: int = 4,
    verbose: bool = True
):
    """
    For y <= threshold_value:
    - Tests every subset of candidate_cols from full set down to min_features
    - Keeps those with all VIF < vif_limit
    - Fits regression on train subset, plots Actual vs Predicted on train and test
    """

    # --- subset train and test where y <= threshold ---
    train_sub = train_df[train_df[target_col] <= threshold_value].copy()
    test_sub  = test_df[test_df[target_col] <= threshold_value].copy()

    y_train = train_sub[target_col].values
    y_test  = test_sub[target_col].values

    p = len(candidate_cols)
    all_valid = []
    print(f"Total covariates: {p}. Searching from {p} down to {min_features}")

    for k in range(p, min_features - 1, -1):
        print(f"\n--- Checking subsets with {k} features ---")
        for combo in itertools.combinations(candidate_cols, k):
            X_train = train_sub[list(combo)]
            X_test  = test_sub[list(combo)]
            vif_df = compute_vif_df(X_train)
            max_vif = float(vif_df["vif"].max())

            if max_vif >= vif_limit:
                continue  # skip multicollinear sets

            mean_vif = float(vif_df["vif"].mean())
            adj_r2 = fit_adj_r2(X_train, y_train)

            # Fit regression on this combo
            model = OLS(y_train, add_constant(X_train, has_constant="add")).fit()
            y_train_pred = model.predict(add_constant(X_train, has_constant="add"))
            y_test_pred  = model.predict(add_constant(X_test, has_constant="add"))

            # store info
            all_valid.append({
                "k": k,
                "combo": combo,
                "max_vif": max_vif,
                "mean_vif": mean_vif,
                "adj_r2": adj_r2,
                "vif_df": vif_df
            })

            # --- scatter plots ---
            plt.figure(figsize=(10,4))

            plt.subplot(1,2,1)
            plt.scatter(y_train, y_train_pred, color="blue", alpha=0.6, s=25)
            min_val = min(y_train.min(), y_train_pred.min())
            max_val = max(y_train.max(), y_train_pred.max())
            plt.plot([min_val, max_val],[min_val,max_val],'k--')
            plt.title(f"Train ({combo})")
            plt.xlabel("Actual"); plt.ylabel("Predicted")
            plt.gca().set_aspect("equal","box")

            plt.subplot(1,2,2)
            plt.scatter(y_test, y_test_pred, color="red", alpha=0.6, s=25)
            min_val = min(y_test.min(), y_test_pred.min())
            max_val = max(y_test.max(), y_test_pred.max())
            plt.plot([min_val, max_val],[min_val,max_val],'k--')
            plt.title(f"Test ({combo})")
            plt.xlabel("Actual"); plt.ylabel("Predicted")
            plt.gca().set_aspect("equal","box")

            plt.suptitle(
                f"Subset size {k} | max VIF={max_vif:.2f}, mean VIF={mean_vif:.2f}, adj R²={adj_r2:.3f}",
                fontsize=10
            )
            plt.tight_layout()
            plt.show()

            if verbose:
                print(f"  Combo: {combo}")
                print(f"  max VIF={max_vif:.3f}, mean VIF={mean_vif:.3f}, adj R²={adj_r2:.4f}")
                print(vif_df.to_string(index=False))
                print()

    if not all_valid:
        print("No subset satisfied the VIF criterion.")
        return None

    # sort best
    all_valid.sort(key=lambda r: (r["max_vif"], r["mean_vif"], -r["adj_r2"]))
    best = all_valid[0]
    print("\nBest subset by VIF + adjusted R²:")
    print(f"  combo={best['combo']}, max VIF={best['max_vif']:.3f}, mean VIF={best['mean_vif']:.3f}, adj R²={best['adj_r2']:.4f}")
    print(best["vif_df"].to_string(index=False))
    return all_valid


In [ ]:
for aquifer, seasons in splits_norm.items():
    for season_label, data in seasons.items():
        print(f"\n=== {aquifer} – {season_label} ===")

        train_df = data["train"]
        test_df  = data["test"]

        num_cols_norm = [c for c in train_df.columns if c != target_col]

        results = search_low_vif_and_plot_regressions(
            train_df=train_df,
            test_df=test_df,
            target_col=target_col,
            candidate_cols=num_cols_norm,
            threshold_value=-0.2,   # your threshold for regression subset
            vif_limit=5.0,
            min_features=4,
            verbose=True
        )


In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:
import os
import pandas as pd
import joblib
import statsmodels.api as sm

def run_loaded_ols_predict(new_df, numerical_cols, categorical_cols, model_dir, aquifer, season_label):
    """
    Load a trained OLS model and predict on new_df.
    """
    # One-hot encode categoricals exactly as before
    new_df = pd.get_dummies(new_df.copy(), columns=categorical_cols, drop_first=True)

    # Load model
    model_path = os.path.join(model_dir, f"{aquifer}_{season_label}_ols_model.pkl")
    model = joblib.load(model_path)
    print(f"Loaded model from: {model_path}")

    # Get expected feature order (excluding constant)
    expected = [c for c in model.params.index if c != "const"]

    # Reindex to match training features, fill missing with zero
    X = new_df.reindex(columns=expected, fill_value=0).astype(float)

    # Add constant
    X_ols = sm.add_constant(X, has_constant="add")

    # Predict
    y_pred = model.predict(X_ols)

    return pd.Series(y_pred, index=new_df.index, name="y_pred")


In [ ]:
model_dir = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/output/"

preds_alluvial = run_loaded_ols_predict(
    new_df=shallow_alluvial,
    numerical_cols=numerical_cols,
    categorical_cols=categorical_cols,
    model_dir=model_dir,
    aquifer="alluvial",
    season_label=5
)


preds_non_alluvial = run_loaded_ols_predict(
    new_df=shallow_non_alluvial,
    numerical_cols=numerical_cols,
    categorical_cols=categorical_cols,
    model_dir=model_dir,
    aquifer="non_alluvial",
    season_label=5
)

In [ ]:
shallow_alluvial

In [ ]:
shallow_non_alluvial

In [ ]:
preds_alluvial

In [ ]:
preds_non_alluvial

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_df = pd.DataFrame()
vif_df["feature"] = X.columns
vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

In [ ]:
threshold = 0.5

# Absolute correlation matrix
corr_matrix = X_train.corr().abs()

# Keep upper triangle only
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Display feature pairs above threshold
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if pd.notnull(upper.loc[row, col]) and upper.loc[row, col] > threshold
]

print("Highly correlated feature pairs (corr > {}):".format(threshold))
for f1, f2, corr_val in high_corr_pairs:
    print(f"{f1}  <-->  {f2}   |  correlation = {corr_val:.3f}")


In [ ]:
# Create a copy of numerical_cols without ET
numerical_cols_no_ET = [col for col in numerical_cols if (col != 'terrain_cluster_1.0' and col !='ET')]
categorical_cols_no_ET = [col for col in categorical_cols if (col != 'terrain_cluster_1.0' and col !='ET')]
# Run the model without ET
run_normalized_ols(train_test_splits, numerical_cols_no_ET, categorical_cols_no_ET, target_col)

In [ ]:
def calculate_vif(X):
    # Keep only numeric columns
    X_numeric = X.select_dtypes(include=[np.number]).astype(float)

    # Drop rows with NaNs/Infs
    X_numeric = X_numeric.replace([np.inf, -np.inf], np.nan).dropna()

    vif_data = pd.DataFrame()
    vif_data["feature"] = X_numeric.columns
    vif_data["VIF"] = [variance_inflation_factor(X_numeric.values, i)
                       for i in range(X_numeric.shape[1])]
    return vif_data

vif_df = calculate_vif(X_train)
print(vif_df)


In [ ]:
# Create a copy of numerical_cols without ET
numerical_cols_no_ET = [col for col in numerical_cols if (col not in ('terrain_cluster_1.0','Area%_SO_1','Area%_SO_2'))]
categorical_cols_no_ET = [col for col in categorical_cols if (col != 'terrain_cluster_1.0' and col !='ET')]
# Run the model without ET
run_normalized_ols(train_test_splits, numerical_cols_no_ET, categorical_cols_no_ET, target_col)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import numpy as np
import pandas as pd

def remove_high_vif_features(X, thresh=10.0):
    """
    Iteratively removes features with VIF greater than the threshold.
    Returns the reduced DataFrame and a DataFrame of final VIF values.
    """
    X = X.select_dtypes(include=[np.number]).astype(float)  # ensure numeric
    dropped_features = []

    while True:
        vif_data = pd.DataFrame()
        vif_data["feature"] = X.columns
        vif_data["VIF"] = [variance_inflation_factor(X.values, i)
                           for i in range(X.shape[1])]

        max_vif = vif_data["VIF"].max()
        if max_vif > thresh:
            drop_feature = vif_data.sort_values("VIF", ascending=False).iloc[0]["feature"]
            print(f"Dropping '{drop_feature}' with VIF={max_vif:.2f}")
            X = X.drop(columns=[drop_feature])
            dropped_features.append(drop_feature)
        else:
            break

    print("\nFinal VIF values:")
    print(vif_data.sort_values("VIF", ascending=False))
    return X, dropped_features, vif_data

In [ ]:
# Suppose X_train is your features DataFrame
X_train_vif, dropped, final_vif = remove_high_vif_features(X_train, thresh=10)

print("\nDropped features:", dropped)

In [ ]:
X_train_vif, dropped, final_vif = remove_high_vif_features(X_train, thresh=10)
X_test_vif = X_test[X_train_vif.columns]  # keep the same columns for test

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_absolute_percentage_error
import matplotlib.pyplot as plt

# Define a function for both Ridge and Lasso
def run_regularized_regression(X_train, X_test, y_train, y_test, model_type='ridge', alpha=1.0):
    if model_type == 'ridge':
        model = Ridge(alpha=alpha)
        title_prefix = "Ridge"
    elif model_type == 'lasso':
        model = Lasso(alpha=alpha)
        title_prefix = "Lasso"
    else:
        raise ValueError("model_type must be 'ridge' or 'lasso'")

    model.fit(X_train, y_train)

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # MAPE
    train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
    test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

    print(f"\n{title_prefix} Regression")
    print(f"Train MAPE: {train_mape:.4f}")
    print(f"Test MAPE:  {test_mape:.4f}")

    # Scatter plots
    plt.figure(figsize=(12, 5))

    # Train scatter
    plt.subplot(1, 2, 1)
    plt.scatter(y_train, y_train_pred, alpha=0.5)
    plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{title_prefix} - Train")

    # Test scatter
    plt.subplot(1, 2, 2)
    plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange')
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{title_prefix} - Test")

    plt.tight_layout()
    plt.show()

# Example usage:
run_regularized_regression(X_train, X_test, y_train, y_test, model_type='ridge', alpha=1.0)
run_regularized_regression(X_train, X_test, y_train, y_test, model_type='lasso', alpha=0.1)


In [ ]:
import seaborn as sns
sns.histplot(sample_weights, bins=30)
plt.title("Distribution of Sample Weights")


In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd

def run_all_regressions_fixed(train_test_splits, numerical_cols, categorical_cols, target_col, alpha=1.0):
    degree = 3
    model_dict = {
        'Ridge': Ridge(alpha=alpha),
        'Lasso': Lasso(alpha=alpha),
        'ElasticNet': ElasticNet(alpha=alpha, l1_ratio=0.5),
        'SVR': SVR(C=1.0),
        'k-NN': KNeighborsRegressor(n_neighbors=5),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
        'MLP': MLPRegressor(hidden_layer_sizes=(100,), max_iter=1000, random_state=42),

    }

    for aquifer, seasons in train_test_splits.items():
        for season_label, data in seasons.items():
            if season_label != 5:
                continue  # Only run for Lag 5

            train_df = data['train'].copy()
            test_df = data['test'].copy()

            # One-hot encode categorical columns
            train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
            test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)
            test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

            # Features
            X_train = train_df[numerical_cols + [col for col in train_df.columns if 'terrain_cluster_' in col]]
            y_train = train_df[target_col]
            X_test = test_df[numerical_cols + [col for col in test_df.columns if 'terrain_cluster_' in col]]
            y_test = test_df[target_col]

            # # Normalize
            # if aquifer == 'alluvial':
            #     X_train[numerical_cols] = alluvial_transformer.fit_transform(X_train[numerical_cols])
            #     X_test[numerical_cols] = alluvial_transformer.transform(X_test[numerical_cols])
            # else:
            #     X_train[numerical_cols] = non_alluvial_transformer.fit_transform(X_train[numerical_cols])
            #     X_test[numerical_cols] = non_alluvial_transformer.transform(X_test[numerical_cols])

            # Inverse sample weights
            sample_weights, _, bin_counts, bin_weights = generate_inverse_sample_weights(y_train)
            print(f"\n===== {aquifer.upper()} – Lag 5 =====")
            print(f"Bin counts: {bin_counts}")
            print(f"Bin weights: {bin_weights}")

            for model_name, model in model_dict.items():
                print(f"\n--- {model_name} Regression ---")

                # Fit model with sample weights if supported
                if model_name in ['Ridge', 'Lasso', 'ElasticNet']:
                    model.fit(X_train, y_train, sample_weight=sample_weights)
                else:
                    model.fit(X_train, y_train)

                # Predict
                y_train_pred = model.predict(X_train)
                y_test_pred = model.predict(X_test)

                # Metrics
                train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
                test_mape = mean_absolute_percentage_error(y_test, y_test_pred)
                r2_train = r2_score(y_train, y_train_pred)
                r2_test = r2_score(y_test, y_test_pred)

                print(f"Train R²: {r2_train:.4f}, Test R²: {r2_test:.4f}")
                print(f"Train MAPE: {train_mape:.4f}, Test MAPE: {test_mape:.4f}")

                # Plot - separate for each model
                fig, axes = plt.subplots(1, 2, figsize=(12, 5))
                for idx, (true, pred, title) in enumerate([
                    (y_train, y_train_pred, 'Train'),
                    (y_test, y_test_pred, 'Test')
                ]):
                    color = 'orange' if title == 'Test' else 'blue'
                    axes[idx].scatter(true, pred, alpha=0.5, color=color)
                    min_val, max_val = min(true.min(), pred.min()), max(true.max(), pred.max())
                    axes[idx].plot([min_val, max_val], [min_val, max_val], 'r--')
                    axes[idx].set_xlabel('Actual')
                    axes[idx].set_ylabel('Predicted')
                    axes[idx].set_title(f'{aquifer} – {season_label} – {model_name} – {title}')
                    axes[idx].axis('equal')

                plt.suptitle(f'{model_name} Regression for {aquifer} – Lag {season_label}')
                plt.tight_layout()
                plt.show()

            # === Polynomial Regression separately ===
            print(f"\n--- Polynomial Regression (degree={degree}) ---")
            poly = PolynomialFeatures(degree=degree)
            X_train_poly = poly.fit_transform(X_train)
            X_test_poly = poly.transform(X_test)

            poly_model = LinearRegression()
            poly_model.fit(X_train_poly, y_train)
            y_train_pred = poly_model.predict(X_train_poly)
            y_test_pred = poly_model.predict(X_test_poly)

            train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)
            r2_train = r2_score(y_train, y_train_pred)
            r2_test = r2_score(y_test, y_test_pred)

            print(f"Train R²: {r2_train:.4f}, Test R²: {r2_test:.4f}")
            print(f"Train MAPE: {train_mape:.4f}, Test MAPE: {test_mape:.4f}")

            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            for idx, (true, pred, title) in enumerate([
                (y_train, y_train_pred, 'Train'),
                (y_test, y_test_pred, 'Test')
            ]):
                color = 'orange' if title == 'Test' else 'blue'
                axes[idx].scatter(true, pred, alpha=0.5, color=color)
                min_val, max_val = min(true.min(), pred.min()), max(true.max(), pred.max())
                axes[idx].plot([min_val, max_val], [min_val, max_val], 'r--')
                axes[idx].set_xlabel('Actual')
                axes[idx].set_ylabel('Predicted')
                axes[idx].set_title(f'{aquifer} – {season_label} – Polynomial – {title}')
                axes[idx].axis('equal')

            plt.suptitle(f'Polynomial Regression (degree={degree}) for {aquifer} – Lag {season_label}')
            plt.tight_layout()
            plt.show()


In [ ]:
run_all_regressions_fixed(train_test_splits, numerical_cols, categorical_cols, target_col, alpha=1.0)

In [ ]:
run_all_regressions_fixed(train_test_splits, numerical_cols, categorical_cols, target_col, alpha=1.0)

In [ ]:
train_test_splits

In [ ]:
# Loop through both aquifer types
for aquifer_type in ['alluvial', 'non_alluvial']:
    print(f"\n===== {aquifer_type.upper()} – LAG 5 =====")

    for split in ['train', 'test']:
        print(f"\n--- {split.upper()} DATA ---")

        df = train_test_splits[aquifer_type][5][split]

        # Ensure 'magnitude_of_slope' exists
        if 'magnitude_of_slope' not in df.columns:
            print(f"'magnitude_of_slope' not found in {split} data for {aquifer_type}")
            continue

        # Get top 5 and bottom 5 rows based on magnitude_of_slope
        top_5 = df.nlargest(5, 'magnitude_of_slope')
        bottom_5 = df.nsmallest(5, 'magnitude_of_slope')

        print("\nTop 5 rows based on 'magnitude_of_slope':")
        print(top_5)

        print("\nBottom 5 rows based on 'magnitude_of_slope':")
        print(bottom_5)

        # Optional: print only timeseries columns
        # If time series has specific columns like 'date', 'month', etc., filter accordingly
        # Otherwise, we print all columns for the selected rows


In [ ]:
import joblib
import os

def load_model_and_columns(model_name, aquifer_type):
    base_path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/output/saved_models"
    feature_set = "deep_columns"  # based on your save pattern
    model = joblib.load(base_path)
    columns = joblib.load(feature_set)

    return model, columns

In [ ]:
base_path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/output/saved_models"
feature_set = "deep_columns"  # based on your save pattern
for aquifer_type in ['Alluvial', 'Non-Alluvial']:
    for model_name in ['Ridge', 'Lasso', 'ElasticNet', 'Random Forest', 'Gradient Boosting', 'MLP', 'XGBoost']:
        filename = f"{model_name}_{aquifer_type}_{feature_set}.pkl"
        full_path = os.path.join(base_path, filename)

        if os.path.exists(full_path):
            print(f"✅ Found: {full_path}")
        else:
            print(f"❌ Missing: {full_path} — Skipping")

In [ ]:
def prepare_data(df, numerical_cols, categorical_cols, required_columns):
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    for col in required_columns:
        if col not in df.columns:
            df[col] = 0
    df = df[required_columns]
    df = df.fillna(df.mean())
    return df

In [ ]:
def evaluate_and_plot(model_name, model, X, y, title_prefix):
    y_pred = model.predict(X)
    mape = mean_absolute_percentage_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    print(f"{title_prefix} - {model_name} | R²: {r2:.4f}, MAPE: {mape:.4f}")

    plt.figure(figsize=(6, 6))
    plt.scatter(y, y_pred, alpha=0.5)
    min_val, max_val = min(y.min(), y_pred.min()), max(y.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{title_prefix} – {model_name}")
    plt.axis("equal")
    plt.tight_layout()
    plt.show()

In [ ]:
model = joblib.load(full_path)

# Debug print
print(f"Loaded type for {model_name}: {type(model)}")

# If it's a string, raise a manual error
if isinstance(model, str):
    raise TypeError(f"Loaded object for {model_name} is a string, not a model.")


In [ ]:
import joblib
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def test_deep_models_on_shallow(train_test_splits, numerical_cols, categorical_cols, target_col):
    base_path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/output/saved_models"
    feature_set = "deep_model"  # Adjust if needed
    results = {}
    key_map = {'Alluvial': 'alluvial', 'Non-Alluvial': 'non_alluvial'}

    for aquifer_type in ['Alluvial', 'Non-Alluvial']:
        test_df = train_test_splits[aquifer_type.lower()][5]['test'].copy()
        X_test = test_df[numerical_cols + categorical_cols]
        y_test = test_df[target_col]
        X_test = X_test.apply(pd.to_numeric, errors='coerce')

        print(f"\n=== Testing on shallow {aquifer_type} data ===")

        results[aquifer_type] = {}

        for model_name in ['Ridge', 'Lasso', 'ElasticNet', 'Random Forest', 'Gradient Boosting', 'MLP', 'XGBoost']:
            filename = f"{model_name}_{aquifer_type}_{feature_set}.pkl"
            full_path = os.path.join(base_path, filename)

            if os.path.exists(full_path):
                try:

                    loaded_obj = joblib.load(full_path)

                    loaded_obj = joblib.load(full_path)
                    if isinstance(loaded_obj, tuple) and len(loaded_obj) == 2:
                        model, model_columns = loaded_obj
                    else:
                        print(f"❌ Error: Unsupported object type loaded: {type(loaded_obj)}")
                        continue
                    y_pred = model.predict(X_test)

                    mae = mean_absolute_error(y_test, y_pred)
                    rmse = mean_squared_error(y_test, y_pred, squared=False)
                    r2 = r2_score(y_test, y_pred)

                    results[aquifer_type][model_name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}

                    print(f"{model_name} on shallow {aquifer_type}: MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")
                except Exception as e:
                    print(f"❌ Error loading/running {model_name} for {aquifer_type}: {e}")
            else:
                print(f"❌ Model not found: {filename}")

    return results

In [ ]:
train_test_splits

In [ ]:
lag5_data

In [ ]:
import joblib

# Load the saved columns used in training
deep_columns_path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/output/saved_models/XGBoost_Alluvial_deep_model.pkl"
model, trained_columns = joblib.load(deep_columns_path)

print("📌 Columns used in trained model:")
print(trained_columns)


In [ ]:
# Assuming lag5_data is your dictionary of data
shallow_df = lag5_data['alluvial'][5]['test']  # or ['non_alluvial']

# One-hot encode if needed
shallow_encoded = pd.get_dummies(shallow_df, columns=categorical_cols, drop_first=True)

print("📌 Columns in shallow test data:")
print(shallow_encoded.columns.tolist())


In [ ]:
trained_set = set(trained_columns)
shallow_set = set(shallow_encoded.columns)

print("✅ Columns in trained model but missing in test:", trained_set - shallow_set)
print("✅ Columns in test but unseen in training:", shallow_set - trained_set)


In [ ]:
lag5_data.keys()

In [ ]:
def align_columns_with_trained(df, trained_columns):
    df = df.copy()

    # Step 1: Rename latitude/longitude and Area%_SO_i columns
    rename_dict = {'latitude': 'Latitude', 'longitude': 'Longitude'}
    for i in range(1, 12):
        rename_dict[f'Area%_SO_{i}'] = f'percent_{i}'
    df.rename(columns=rename_dict, inplace=True)

    # Step 2: Drop columns not used during training
    df = df[[col for col in df.columns if col in trained_columns]]

    # Step 3: Add missing columns (fill with 0)
    for col in trained_columns:
        if col not in df.columns:
            df[col] = 0

    # Step 4: Reorder to match trained model
    df = df[trained_columns]

    return df

# Inference loop
base_path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/output/saved_models"
feature_set = "deep_model"
model_names = ['Ridge', 'Lasso', 'ElasticNet', 'Random Forest', 'Gradient Boosting', 'MLP', 'XGBoost']

for aquifer_type in ['Alluvial', 'Non_Alluvial']:
    print(f"\n=== Inference for {aquifer_type.upper()} Shallow Data ===")

    shallow_sets = lag5_data[aquifer_type.lower()][5]
    train_df = shallow_sets['train']
    test_df = shallow_sets['test']
    combined_df = pd.concat([train_df, test_df], ignore_index=True)

    for model_name in model_names:
        full_path = os.path.join(base_path, f"{model_name}_{aquifer_type}_{feature_set}.pkl")

        try:
            model,trained_columns = joblib.load(full_path)

            print(f"\n✅ Loaded {model_name} for {aquifer_type}")

            for label, df in [('Train', train_df), ('Test', test_df), ('Combined', combined_df)]:
                aligned_df = align_columns_with_trained(df, trained_columns)
                y_pred = model.predict(aligned_df)
                print(f"{label} Prediction shape: {y_pred.shape}")
        except Exception as e:
            print(f"❌ Error with {model_name} for {aquifer_type}: {e}")

In [ ]:
import matplotlib.pyplot as plt
import os
import joblib
import pandas as pd

def align_columns_with_trained(df, trained_columns):
    df = df.copy()
    rename_dict = {'latitude': 'Latitude', 'longitude': 'Longitude'}
    for i in range(1, 12):
        rename_dict[f'Area%_SO_{i}'] = f'percent_{i}'
    df.rename(columns=rename_dict, inplace=True)
    df = df[[col for col in df.columns if col in trained_columns]]
    for col in trained_columns:
        if col not in df.columns:
            df[col] = 0
    df = df[trained_columns]
    return df

def plot_actual_vs_pred(y_true, y_pred, title):
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.5, edgecolor='k')
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--')
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.grid(True)
    plt.axis('equal')
    plt.tight_layout()
    plt.show()

# === Main Loop ===
base_path = f"{WORKSPACE_ROOT}/22. Training_deep_five_years/output/saved_models"
feature_set = "deep_model"
model_names = ['Ridge', 'Lasso', 'ElasticNet', 'Random Forest', 'Gradient Boosting', 'MLP', 'XGBoost']
splits = ['Train', 'Test', 'Combined']

for aquifer_type in ['Alluvial', 'Non_Alluvial']:
    print(f"\n=== Inference & Visualization for {aquifer_type.upper()} ===")
    shallow_sets = lag5_data[aquifer_type.lower()][5]
    split_data = {
        'Train': shallow_sets['train'],
        'Test': shallow_sets['test'],
        'Combined': pd.concat([shallow_sets['train'], shallow_sets['test']], ignore_index=True)
    }

    for model_name in model_names:
        model_path = os.path.join(base_path, f"{model_name}_{aquifer_type}_{feature_set}.pkl")

        try:
            model, trained_columns = joblib.load(model_path)
            print(f"\n✅ Loaded {model_name} for {aquifer_type}")

            for split_label in splits:
                df = split_data[split_label]
                aligned_df = align_columns_with_trained(df, trained_columns)
                y_true = df[target_col].values
                y_pred = model.predict(aligned_df)

                plot_actual_vs_pred(y_true, y_pred,
                    f"{model_name} – {aquifer_type} – {split_label} Data")

        except Exception as e:
            print(f"❌ Error with {model_name} for {aquifer_type}: {e}")


In [ ]:
train_df_aligned = align_columns_with_trained(lag5_data['alluvial'][5]['train'], trained_columns)
test_df_aligned = align_columns_with_trained(lag5_data['alluvial'][5]['test'], trained_columns)
combined_df_aligned = align_columns_with_trained(
    pd.concat([lag5_data['alluvial'][5]['train'], lag5_data['alluvial'][5]['test']], ignore_index=True),
    trained_columns
)

In [ ]:
shallow_encoded

In [ ]:
test_deep_models_on_shallow(shallow_encoded, numerical_cols, categorical_cols, target_col)

In [ ]:
plt.scatter(y_train, y_train_pred, c=sample_weights, cmap='coolwarm', alpha=0.6)
plt.colorbar(label='Sample Weight')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Train: Prediction vs Actual with Weight Coloring')

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error
import matplotlib.pyplot as plt
import numpy as np

# Store models in a dictionary
regression_models = {
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "Polynomial": make_pipeline(PolynomialFeatures(degree=2), Ridge(alpha=1.0)),
    "SVR": make_pipeline(StandardScaler(), SVR(C=1.0, epsilon=0.1)),
    "k-NN": KNeighborsRegressor(n_neighbors=5),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "GBR": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42),
    "MLP": MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
}



# Dictionary to store results
model_results = {}

# Fit models and store predictions
for name, model in regression_models.items():
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
    test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

    model_results[name] = {
        "train_pred": y_train_pred,
        "test_pred": y_test_pred,
        "train_mape": train_mape,
        "test_mape": test_mape
    }

    # Plot scatter
    plt.figure(figsize=(6, 6))
    plt.scatter(y_train, y_train_pred, alpha=0.5, label='Train')
    plt.scatter(y_test, y_test_pred, alpha=0.5, color='orange', label='Test')
    min_val = min(np.min(y_train), np.min(y_test), np.min(y_train_pred), np.min(y_test_pred))
    max_val = max(np.max(y_train), np.max(y_test), np.max(y_train_pred), np.max(y_test_pred))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title(f'{name} Regression')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

model_results.keys()


In [ ]:
df = pd.DataFrame({'y_true': y_train, 'y_pred': y_train_pred, 'bin': bin_labels})
print(df.groupby('bin')[['y_true', 'y_pred']].agg(['mean', 'std']))

In [ ]:
print(f"Train y range for {aquifer}-{season_label}: min={y_train.min()}, max={y_train.max()}")
print(f"Bin counts: {bin_counts}")
